In [5]:
import sys
import numpy as np
from scipy import stats
from scipy.optimize import least_squares
from scipy.stats import poisson, binom, bernoulli, lognorm, beta, genpareto, norm, uniform
import matplotlib
matplotlib.use('Qt5Agg')
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import pandas as pd
from matplotlib.ticker import FuncFormatter
import warnings
from tabulate import tabulate
from PyQt5 import QtCore, QtGui, QtWidgets
from PyQt5.QtCore import QThread, pyqtSignal
import json
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import cm
import io
from reportlab.lib.utils import ImageReader
import traceback
import uuid
import copy
import shap
import sklearn.linear_model

def ajustar_distribuciones(losses):
    ajustes, pruebas = {}, []

    s, _, scale = stats.lognorm.fit(losses, floc=0)
    D, p = stats.kstest(losses, 'lognorm', args=(s, 0, scale))
    ajustes['LogNormal'] = {'s': round(s,4), 'scale': round(scale,2),
                            'loc': 0, 'KS_D': round(D,4), 'KS_p': round(p,4)}
    pruebas.append(('LogNormal', D))

    mu, sigma = stats.norm.fit(losses)
    D, p = stats.kstest(losses, 'norm', args=(mu, sigma))
    ajustes['Normal'] = {'mu': round(mu,2), 'sigma': round(sigma,2),
                         'KS_D': round(D,4), 'KS_p': round(p,4)}
    pruebas.append(('Normal', D))

    a, loc_g, scale_g = stats.gamma.fit(losses, floc=0)
    D, p = stats.kstest(losses, 'gamma', args=(a, 0, scale))
    ajustes['Gamma'] = {'a': round(a,4), 'scale': round(scale_g,2),
                        'loc': 0, 'KS_D': round(D,4), 'KS_p': round(p,4)}
    pruebas.append(('Gamma', D))

    best = min(pruebas, key=lambda x: x[1])[0]
    return ajustes, best

# Configuramos Seaborn y Matplotlib
sns.set(style="whitegrid")

# Ignoramos advertencias de runtime
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Función para formatear números en formato contable personalizado
def currency_format(value):
    """Formatea un número en formato monetario con separador de miles y decimales personalizados."""
    return '${:,.0f}'.format(value).replace(',', 'X').replace('.', ',').replace('X', '.')

# Función para formatear porcentajes sin decimales
def percentage_format(value):
    """Formatea un número entre 0 y 1 como porcentaje sin decimales."""
    return '{:,.0f}%'.format(value * 100).replace('.', ',')

# Funciones para formatear los ejes en los gráficos
def currency_formatter(x, pos):
    """Función de formateo para los ejes que muestra valores monetarios."""
    return currency_format(x)

def percentage_formatter(x, pos):
    """Función de formateo para los ejes que muestra porcentajes."""
    return percentage_format(x)

# Funciones para obtener parámetros de distribuciones

def obtener_parametros_normal(minimo, mas_probable, maximo):
    """Calcula los parámetros mu y sigma para la distribución Normal utilizando mínimos y máximos."""
    if minimo >= maximo:
        raise ValueError("El valor mínimo debe ser menor que el máximo.")
    mu = mas_probable
    sigma = (maximo - minimo) / (2 * 3)
    return mu, sigma

def obtener_parametros_lognormal(minimo, mas_probable, maximo):
    """Calcula los parámetros mu y sigma para la distribución Lognormal."""
    if minimo <= 0 or mas_probable <= 0 or maximo <= 0 or not (minimo < mas_probable < maximo):
        raise ValueError("Los valores deben ser positivos y cumplir mínimo < más probable < máximo.")

    P1 = 0.10
    P2 = 0.90

    def equations(p):
        mu, sigma = p
        if sigma <= 0:
            return [np.inf, np.inf, np.inf]
        eq1 = np.exp(mu - sigma**2) - mas_probable
        eq2 = stats.lognorm.cdf(minimo, s=sigma, scale=np.exp(mu)) - P1
        eq3 = stats.lognorm.cdf(maximo, s=sigma, scale=np.exp(mu)) - P2
        return [eq1, eq2, eq3]

    mu_initial = np.log(mas_probable)
    sigma_initial = 1.0

    result = least_squares(equations, x0=[mu_initial, sigma_initial],
                           bounds=([-np.inf, 1e-6], [np.inf, np.inf]))
    mu, sigma = result.x

    if not result.success or sigma <= 0:
        raise ValueError("No se pudieron estimar los parámetros de la distribución Lognormal.")

    return mu, sigma

def obtener_parametros_pert(minimo, mas_probable, maximo):
    """Calcula los parámetros alpha y beta para la distribución PERT."""
    if maximo == minimo:
        raise ValueError("El valor máximo y mínimo no pueden ser iguales para la distribución PERT.")
    if not (minimo <= mas_probable <= maximo):
        raise ValueError("Los valores deben cumplir que mínimo <= más probable <= máximo.")
    alpha = 1 + 4 * ((mas_probable - minimo) / (maximo - minimo))
    beta_param = 1 + 4 * ((maximo - mas_probable) / (maximo - minimo))
    return alpha, beta_param

def obtener_parametros_gpd(minimo, mas_probable, maximo):
    """Calcula los parámetros xi y beta para la distribución Generalizada de Pareto."""
    if minimo <= 0 or mas_probable <= minimo or mas_probable >= maximo:
        raise ValueError("Los valores deben cumplir mínimo > 0 y mínimo < más probable < máximo.")

    mu = minimo

    p1 = 0.5
    p2 = 0.95

    x1 = mas_probable
    x2 = maximo

    def objective(p):
        xi, beta = p
        if beta <= 0:
            return [np.inf, np.inf]
        try:
            cdf1 = 1 - (1 + xi * (x1 - mu) / beta) ** (-1 / xi)
            cdf2 = 1 - (1 + xi * (x2 - mu) / beta) ** (-1 / xi)
        except (ZeroDivisionError, OverflowError, ValueError):
            return [np.inf, np.inf]
        return [cdf1 - p1, cdf2 - p2]

    xi_initial = 0.1
    beta_initial = x1 - mu

    result = least_squares(objective, x0=[xi_initial, beta_initial],
                           bounds=([-np.inf, 1e-6], [np.inf, np.inf]))
    xi, beta = result.x

    if not result.success or beta <= 0:
        raise ValueError("No se pudieron estimar los parámetros de la GPD.")

    return xi, beta, mu

def obtener_parametros_uniforme(minimo, maximo):
    """Configura los parámetros para la distribución Uniforme."""
    if minimo >= maximo:
        raise ValueError("El valor mínimo debe ser menor que el máximo.")
    return minimo, maximo - minimo

# Funciones para generar y validar distribuciones de severidad y frecuencia
def generar_distribucion_severidad(opcion, minimo, mas_probable, maximo,
                                 input_method='min_mode_max', params_direct=None):
    """
    Genera un objeto de distribución de severidad de scipy.stats.

    Args:
        opcion (int): El índice de la distribución (1:Normal, 2:LogNormal, etc.).
        minimo (float): Valor mínimo (usado si input_method es 'min_mode_max').
        mas_probable (float): Valor más probable (usado si input_method es 'min_mode_max').
        maximo (float): Valor máximo (usado si input_method es 'min_mode_max').
        input_method (str, optional): 'min_mode_max' o 'direct'. Default 'min_mode_max'.
        params_direct (dict, optional): Diccionario con parámetros directos si input_method es 'direct'.
                                         Ej. Lognormal: {'s': _, 'scale': _, 'loc': _}
                                         Ej. GPD: {'c': _, 'scale': _, 'loc': _}
                                         Default None.
    Returns:
        scipy.stats.rv_frozen: Objeto de distribución congelado.
    Raises:
        ValueError: Si los parámetros son inválidos o el método es incorrecto.
    """
    try:
        distribucion_severidad = None
        dist_nombre = "" # Para mensajes de error

        # --- Lógica para Lognormal (opcion == 2) ---
        if opcion == 2:
            dist_nombre = "Lognormal"
            if input_method == 'direct':
                if not params_direct or 's' not in params_direct or 'scale' not in params_direct:
                    raise ValueError("Parámetros directos 's' y 'scale' requeridos para Lognormal.")
                s = params_direct['s']
                scale = params_direct['scale']
                loc = params_direct.get('loc', 0) # loc es opcional, default 0
                if s <= 0: raise ValueError("Shape (s) debe ser positivo.")
                if scale <= 0: raise ValueError("Scale debe ser positivo.")
                distribucion_severidad = lognorm(s=s, scale=scale, loc=loc)
            elif input_method == 'min_mode_max':
                 # Validar que Min/Mode/Max no sean None si se usa este método
                if minimo is None or mas_probable is None or maximo is None:
                     raise ValueError("Min/Mode/Max requeridos para Lognormal con este método.")
                mu, sigma = obtener_parametros_lognormal(minimo, mas_probable, maximo) # Llama a la función antigua
                distribucion_severidad = lognorm(s=sigma, scale=np.exp(mu))
            else:
                raise ValueError(f"Método de entrada '{input_method}' no válido para {dist_nombre}.")

        # --- Lógica para GPD (opcion == 4) ---
        elif opcion == 4:
            dist_nombre = "GPD (Pareto Generalizada)"
            if input_method == 'direct':
                if not params_direct or 'c' not in params_direct or 'scale' not in params_direct or 'loc' not in params_direct:
                     raise ValueError("Parámetros directos 'c', 'scale' y 'loc' requeridos para GPD.")
                c = params_direct['c']
                scale = params_direct['scale']
                loc = params_direct['loc']
                if scale <= 0: raise ValueError("Scale (beta) debe ser positivo.")
                # Podrían necesitarse más validaciones (ej. sobre c y loc)
                distribucion_severidad = genpareto(c=c, scale=scale, loc=loc)
            elif input_method == 'min_mode_max':
                # Validar que Min/Mode/Max no sean None si se usa este método
                if minimo is None or mas_probable is None or maximo is None:
                     raise ValueError("Min/Mode/Max requeridos para GPD con este método.")
                xi, beta_param, mu = obtener_parametros_gpd(minimo, mas_probable, maximo) # Llama a la función antigua
                distribucion_severidad = genpareto(c=xi, scale=beta_param, loc=mu)
            else:
                raise ValueError(f"Método de entrada '{input_method}' no válido para {dist_nombre}.")

        # --- Lógica para otras distribuciones (usan solo Min/Mode/Max implícitamente) ---
        elif opcion == 5: # Uniforme
             dist_nombre = "Uniforme"
             if minimo is None or maximo is None: raise ValueError("Min/Max requeridos para Uniforme.")
             loc, scale_unif = obtener_parametros_uniforme(minimo, maximo)
             distribucion_severidad = uniform(loc=loc, scale=scale_unif)
        elif opcion == 1: # Normal
             dist_nombre = "Normal"
             if minimo is None or mas_probable is None or maximo is None: raise ValueError("Min/Mode/Max requeridos para Normal.")
             mu, sigma = obtener_parametros_normal(minimo, mas_probable, maximo)
             distribucion_severidad = norm(loc=mu, scale=sigma)
        elif opcion == 3: # PERT (Beta)
             dist_nombre = "PERT (Beta)"
             if minimo is None or mas_probable is None or maximo is None: raise ValueError("Min/Mode/Max requeridos para PERT.")
             a, b = obtener_parametros_pert(minimo, mas_probable, maximo)
             distribucion_severidad = beta(a, b, loc=minimo, scale=maximo - minimo)
        else:
            raise ValueError(f"Opción de distribución de severidad desconocida: {opcion}")

        # Comprobación final
        if distribucion_severidad is None:
             raise ValueError(f"No se pudo crear la distribución {dist_nombre}.")

        return distribucion_severidad

    except ValueError as ve:
        # Re-lanzar el error para que sea capturado por guardar_evento
        raise ValueError(f"Error al generar {dist_nombre}: {ve}")
    except Exception as e:
        # Capturar otros errores inesperados
        raise Exception(f"Error inesperado al generar {dist_nombre}: {e}")

def generar_distribucion_frecuencia(opcion, tasa=None, num_eventos_posibles=None, probabilidad_exito=None):
    try:
        if opcion == 1:
            distribucion_frecuencia = poisson(mu=tasa)
        elif opcion == 2:
            distribucion_frecuencia = binom(n=num_eventos_posibles, p=probabilidad_exito)
        elif opcion == 3:
            distribucion_frecuencia = bernoulli(p=probabilidad_exito)
        return distribucion_frecuencia
    except ValueError as ve:
        raise ve
    except Exception as e:
        raise e

def ordenar_eventos_por_dependencia(eventos_riesgo):
    id_a_evento = {evento['id']: evento for evento in eventos_riesgo}
    visitados = set()
    stack = []

    def dfs(evento_id):
        visitados.add(evento_id)
        evento = id_a_evento[evento_id]

        # Buscar los hijos (eventos que dependen de este)
        hijos_ids = []
        for ev in eventos_riesgo:
            # Verificar en la nueva estructura de vínculos
            if 'vinculos' in ev:
                for vinculo in ev['vinculos']:
                    if vinculo['id_padre'] == evento_id:
                        hijos_ids.append(ev['id'])
                        break
            # Compatibilidad con formato antiguo
            elif 'eventos_padres' in ev and evento_id in ev.get('eventos_padres', []):
                hijos_ids.append(ev['id'])

        for hijo_id in hijos_ids:
            if hijo_id not in visitados:
                dfs(hijo_id)
        stack.append(evento_id)

    for evento in eventos_riesgo:
        if evento['id'] not in visitados:
            dfs(evento['id'])

    stack.reverse()
    return stack

# Funciones para generar la LDA y mostrar resultados
def generar_lda_con_secuencialidad(eventos_riesgo, num_simulaciones=10000):
    id_a_evento = {evento['id']: evento for evento in eventos_riesgo}
    id_a_index = {evento['id']: idx for idx, evento in enumerate(eventos_riesgo)}
    num_eventos = len(eventos_riesgo)
    perdidas_totales = np.zeros(num_simulaciones)
    frecuencias_totales = np.zeros(num_simulaciones)
    perdidas_por_evento = [np.zeros(num_simulaciones) for _ in range(num_eventos)]
    frecuencias_por_evento = [np.zeros(num_simulaciones, dtype=int) for _ in range(num_eventos)]

    # Ordenar eventos para procesar padres antes que hijos
    orden_eventos_ids = ordenar_eventos_por_dependencia(eventos_riesgo)

    for evento_id in orden_eventos_ids:
        idx = id_a_index[evento_id]
        evento = id_a_evento[evento_id]
        dist_sev = evento['dist_severidad']
        dist_freq = evento['dist_frecuencia']

        # Compatibilidad: verificar si usa la nueva estructura o la antigua
        if 'vinculos' in evento and evento['vinculos']:
            vinculos = evento['vinculos']

            # Agrupar vínculos por tipo
            vinculos_por_tipo = {'AND': [], 'OR': [], 'EXCLUYE': []}
            for vinculo in vinculos:
                tipo = vinculo['tipo']
                id_padre = vinculo['id_padre']
                vinculos_por_tipo[tipo].append(id_padre)

            # Inicializar condiciones para cada tipo
            condicion_and = np.ones(num_simulaciones, dtype=bool)
            condicion_or = np.zeros(num_simulaciones, dtype=bool)
            condicion_excluye = np.ones(num_simulaciones, dtype=bool)

            # Procesar vínculos AND (todos deben ocurrir)
            for padre_id in vinculos_por_tipo['AND']:
                padre_idx = id_a_index[padre_id]
                condicion_and = condicion_and & (frecuencias_por_evento[padre_idx] > 0)

            # Procesar vínculos OR (al menos uno debe ocurrir)
            for padre_id in vinculos_por_tipo['OR']:
                padre_idx = id_a_index[padre_id]
                condicion_or = condicion_or | (frecuencias_por_evento[padre_idx] > 0)

            # Procesar vínculos EXCLUYE (ninguno debe ocurrir)
            for padre_id in vinculos_por_tipo['EXCLUYE']:
                padre_idx = id_a_index[padre_id]
                condicion_excluye = condicion_excluye & (frecuencias_por_evento[padre_idx] == 0)

            # Combinar condiciones según los tipos presentes
            condicion_final = np.ones(num_simulaciones, dtype=bool)

            if vinculos_por_tipo['AND']:
                condicion_final = condicion_final & condicion_and

            if vinculos_por_tipo['OR']:
                condicion_final = condicion_final & condicion_or

            if vinculos_por_tipo['EXCLUYE']:
                condicion_final = condicion_final & condicion_excluye

            # Simular frecuencia solo para las condiciones que se cumplen
            muestras_frecuencia = np.zeros(num_simulaciones, dtype=int)
            indices_a_simular = np.where(condicion_final)[0]

            if len(indices_a_simular) > 0:
                muestras_frecuencia_simuladas = dist_freq.rvs(size=len(indices_a_simular))
                muestras_frecuencia_simuladas = np.maximum(muestras_frecuencia_simuladas, 0).astype(int)
                muestras_frecuencia[indices_a_simular] = muestras_frecuencia_simuladas

        elif 'eventos_padres' in evento and evento['eventos_padres']:
            # Formato antiguo: usar tipo_dependencia único para todos los padres
            eventos_padres = evento.get('eventos_padres', [])
            tipo_dependencia = evento.get('tipo_dependencia', 'AND')

            if eventos_padres:
                # Verificar condiciones según tipo de dependencia
                ocurrencias_padres = np.vstack(
                    [frecuencias_por_evento[id_a_index[padre_id]] > 0 for padre_id in eventos_padres]
                )

                if tipo_dependencia == 'AND':
                    condicion_padres = np.all(ocurrencias_padres, axis=0)
                elif tipo_dependencia == 'OR':
                    condicion_padres = np.any(ocurrencias_padres, axis=0)
                elif tipo_dependencia in ['MUTEX', 'EXCLUYE']:
                    condicion_padres = ~np.any(ocurrencias_padres, axis=0)
                else:
                    condicion_padres = np.ones(num_simulaciones, dtype=bool)

                muestras_frecuencia = np.zeros(num_simulaciones, dtype=int)
                indices_a_simular = np.where(condicion_padres)[0]

                if len(indices_a_simular) > 0:
                    muestras_frecuencia_simuladas = dist_freq.rvs(size=len(indices_a_simular))
                    muestras_frecuencia_simuladas = np.maximum(muestras_frecuencia_simuladas, 0).astype(int)
                    muestras_frecuencia[indices_a_simular] = muestras_frecuencia_simuladas
            else:
                # Evento sin padres, simular normalmente
                muestras_frecuencia = dist_freq.rvs(size=num_simulaciones)
                muestras_frecuencia = np.maximum(muestras_frecuencia, 0).astype(int)
        else:
            # Evento sin dependencias, simular normalmente
            muestras_frecuencia = dist_freq.rvs(size=num_simulaciones)
            muestras_frecuencia = np.maximum(muestras_frecuencia, 0).astype(int)

        # Guardar frecuencias y calcular pérdidas
        frecuencias_por_evento[idx] = muestras_frecuencia
        frecuencias_totales += muestras_frecuencia

        # Generar pérdidas
        total_eventos = muestras_frecuencia.sum()
        if total_eventos > 0:
            total_perdidas_evento = dist_sev.rvs(size=total_eventos)
            total_perdidas_evento = np.maximum(total_perdidas_evento, 0)

            # Dividir las pérdidas en sub-arrays para cada simulación
            perdidas_evento = np.zeros(num_simulaciones)
            indices_no_cero = np.where(muestras_frecuencia > 0)[0]
            counts = muestras_frecuencia[indices_no_cero]

            if len(counts) > 0:  # Verificar que hay elementos no cero
                split_indices = np.cumsum(counts)[:-1]
                perdidas_split = np.split(total_perdidas_evento, split_indices)
                perdidas_evento[indices_no_cero] = np.array([arr.sum() for arr in perdidas_split])

            perdidas_por_evento[idx] = perdidas_evento
            perdidas_totales += perdidas_evento
        else:
            perdidas_por_evento[idx] = np.zeros(num_simulaciones)

    return perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento

# Función para mostrar el resumen ejecutivo con datos en negrita
def obtener_resumen_ejecutivo_texto(media, desviacion_estandar, var_90, opvar_99, opvar, min_freq_total, mode_freq_total, max_freq_total):
    texto = "\nResumen Ejecutivo de Resultados de Pérdidas Agregadas:\n"
    texto += f"Media de Pérdidas Agregadas: {currency_format(round(media))}\n"
    texto += f"Desviación Estándar: {currency_format(round(desviacion_estandar))}\n"
    texto += f"VaR al 90%: {currency_format(round(var_90))}\n"
    texto += f"OpVaR al 99%: {currency_format(round(opvar_99))}\n"
    texto += f"Pérdida Esperada más allá del OpVaR 99%: {currency_format(round(opvar))}\n"
    texto += f"Número mínimo de eventos materializados: {min_freq_total}\n"
    texto += f"Número más probable de eventos materializados: {mode_freq_total}\n"
    texto += f"Número máximo de eventos materializados: {max_freq_total}\n"
    return texto

# Función para mostrar tablas de percentiles con formato
def obtener_tabla_percentiles_texto(percentiles_df, titulo):
    texto = f"\n{titulo}\n\n"
    texto += tabulate(percentiles_df, headers='keys', tablefmt='fancy_grid', showindex=False)
    texto += "\n"
    return texto

class SimulacionThread(QThread):
    # Definir señales
    progreso_actualizado = pyqtSignal(int)
    simulacion_completada = pyqtSignal(object, object, object, object, object)
    error_ocurrido = pyqtSignal(str)

    def __init__(self, eventos_riesgo, num_simulaciones, generar_reporte, pdf_filename):
        super().__init__()
        self.eventos_riesgo = eventos_riesgo
        self.num_simulaciones = num_simulaciones
        self.generar_reporte = generar_reporte
        self.pdf_filename = pdf_filename
        self.is_running = True

    def run(self):
        try:
            total = self.num_simulaciones
            num_chunks = 100
            chunk_size = total // num_chunks

            # Inicializar los arrays de resultados completos
            perdidas_totales = np.zeros(self.num_simulaciones)
            frecuencias_totales = np.zeros(self.num_simulaciones)
            perdidas_por_evento = None
            frecuencias_por_evento = None

            for i in range(num_chunks):
                if not self.is_running:
                    break
                start = i * chunk_size
                end = start + chunk_size if i < num_chunks - 1 else total

                # Ejecutar una parte de la simulación
                resultados = generar_lda_con_secuencialidad(
                    self.eventos_riesgo,
                    num_simulaciones=end - start
                )

                # Actualizar los resultados acumulados
                # Actualizar perdidas_totales y frecuencias_totales
                perdidas_totales[start:end] = resultados[0]
                frecuencias_totales[start:end] = resultados[1]

                # Inicializar perdidas_por_evento y frecuencias_por_evento la primera vez
                if perdidas_por_evento is None:
                    num_eventos = len(resultados[2])
                    perdidas_por_evento = [np.zeros(self.num_simulaciones) for _ in range(num_eventos)]
                    frecuencias_por_evento = [np.zeros(self.num_simulaciones, dtype=int) for _ in range(num_eventos)]

                # Acumular perdidas_por_evento y frecuencias_por_evento
                for idx in range(len(perdidas_por_evento)):
                    perdidas_por_evento[idx][start:end] = resultados[2][idx]
                    frecuencias_por_evento[idx][start:end] = resultados[3][idx]

                # Emitir señal de progreso
                progreso = int((i + 1) * 100 / num_chunks)
                self.progreso_actualizado.emit(progreso)

            if not self.is_running:
                return

            self.simulacion_completada.emit(
                perdidas_totales,
                frecuencias_totales,
                perdidas_por_evento,
                frecuencias_por_evento,
                self.eventos_riesgo
            )
        except Exception as e:
            self.error_ocurrido.emit(str(e))

    def stop(self):
        self.is_running = False

class Scenario:
    def __init__(self, nombre, descripcion=""):
        self.nombre = nombre
        self.descripcion = descripcion
        self.eventos_riesgo = []

    def to_dict(self):
        return {
            'nombre': self.nombre,
            'descripcion': self.descripcion,
            'eventos_riesgo': [evento.copy() for evento in self.eventos_riesgo]
        }

    @staticmethod
    def from_dict(data):
        scenario = Scenario(data['nombre'], data.get('descripcion', ''))
        scenario.eventos_riesgo = data.get('eventos_riesgo', [])
        return scenario

# Clase para la interfaz gráfica usando PyQt5
class RiskLabApp(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Risk Lab")
        self.APP_VERSION = "1.9"
        self.setWindowIcon(QtGui.QIcon("icon.png"))
        self.eventos_riesgo = []
        self.scenarios = []
        self.current_scenario = None
        self.generar_reporte = False
        self.pdf_filename = 'reporte_simulacion.pdf'
        self.setup_ui()
        self.apply_stylesheet()

    def setup_ui(self):
        # Widget central con pestañas
        self.central_widget = QtWidgets.QTabWidget()
        self.setCentralWidget(self.central_widget)

        # Pestaña de Simulación
        self.config_tab = QtWidgets.QWidget()
        self.setup_config_tab()
        self.central_widget.addTab(self.config_tab, "Simulación")

        # Pestaña de Escenarios
        self.scenarios_tab = QtWidgets.QWidget()
        self.setup_scenarios_tab()
        self.central_widget.addTab(self.scenarios_tab, "Escenarios")

        # Pestaña de Resultados
        self.results_tab = QtWidgets.QWidget()
        self.setup_results_tab()
        self.central_widget.addTab(self.results_tab, "Resultados")

        # Menú
        menu_bar = self.menuBar()
        # Menú Archivo
        archivo_menu = menu_bar.addMenu("Archivo")

        # Agregar acción Nueva Simulación
        nuevo_action = QtWidgets.QAction("Nueva Simulación", self)
        archivo_menu.addAction(nuevo_action)
        nuevo_action.triggered.connect(self.nueva_simulacion)

        guardar_config_action = QtWidgets.QAction("Guardar Simulación", self)
        cargar_config_action = QtWidgets.QAction("Cargar Simulación", self)
        exportar_pdf_action = QtWidgets.QAction("Exportar Reporte", self)
        salir_action = QtWidgets.QAction("Salir", self)

        archivo_menu.addAction(guardar_config_action)
        archivo_menu.addAction(cargar_config_action)
        archivo_menu.addAction(exportar_pdf_action)
        archivo_menu.addSeparator()
        archivo_menu.addAction(salir_action)

        guardar_config_action.triggered.connect(self.guardar_configuracion)
        cargar_config_action.triggered.connect(self.cargar_configuracion)
        exportar_pdf_action.triggered.connect(self.exportar_a_pdf)
        salir_action.triggered.connect(self.close)

        # Menú Ayuda
        ayuda_menu = menu_bar.addMenu("Ayuda")
        acerca_de_action = QtWidgets.QAction("Acerca de", self)
        ayuda_menu.addAction(acerca_de_action)

        acerca_de_action.triggered.connect(self.mostrar_acerca_de)

    def nueva_simulacion(self):
        respuesta = QtWidgets.QMessageBox.question(
            self,
            "Nueva Simulación",
            "¿Estás seguro de que deseas iniciar una nueva simulación? Se perderán todos los datos actuales.",
            QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No
        )
        if respuesta == QtWidgets.QMessageBox.Yes:
            # Limpiar eventos de riesgo
            self.eventos_riesgo.clear()
            self.eventos_table.setRowCount(0)

            # Limpiar escenarios
            self.scenarios.clear()
            self.scenarios_table.setRowCount(0)
            self.current_scenario = None
            self.selected_scenario_label.setText("Ninguno")

            # Limpiar resultados de simulación
            self.resultados_text_edit.clear()
            self.graficos_tab_widget.clear()

            # Restablecer número de simulaciones a un valor por defecto si lo deseas
            self.num_simulaciones_var.setText("10000")
            self.num_simulaciones_var_escenarios.setText("10000")

            # Notificar al usuario que se ha iniciado una nueva simulación
            QtWidgets.QMessageBox.information(self, "Nueva Simulación", "Se ha iniciado una nueva simulación.")

    def mostrar_acerca_de(self):
        mensaje = f"""
        <h2>Risk Lab</h2>
        <p>Versión {self.APP_VERSION}</p>
        <p>Aplicación creada por Hernán Zvirblis.</p>
        <p>Esta aplicación es una herramienta para la simulación de riesgos utilizando el método de Monte Carlo.</p>
        """
        QtWidgets.QMessageBox.about(self, "Acerca de", mensaje)

    def setup_config_tab(self):
        layout = QtWidgets.QVBoxLayout(self.config_tab)

        # Título
        titulo_label = QtWidgets.QLabel("Simulación Avanzada de Riesgos")
        titulo_label.setFont(QtGui.QFont("Helvetica", 20, QtGui.QFont.Bold))
        titulo_label.setAlignment(QtCore.Qt.AlignCenter)
        layout.addWidget(titulo_label)

        # Número de simulaciones
        simulaciones_layout = QtWidgets.QHBoxLayout()
        simulaciones_label = QtWidgets.QLabel("Número de simulaciones:")
        self.num_simulaciones_var = QtWidgets.QLineEdit("10000")
        self.num_simulaciones_var.setFixedWidth(100)
        simulaciones_layout.addWidget(simulaciones_label)
        simulaciones_layout.addWidget(self.num_simulaciones_var)
        simulaciones_layout.addStretch()
        layout.addLayout(simulaciones_layout)

        # Botón para agregar evento
        agregar_evento_button = QtWidgets.QPushButton("Agregar Evento de Riesgo")
        agregar_evento_button.setIcon(QtGui.QIcon.fromTheme("list-add"))
        agregar_evento_button.clicked.connect(self.agregar_evento_popup)
        layout.addWidget(agregar_evento_button)

        # Lista de eventos
        eventos_layout = QtWidgets.QHBoxLayout()
        eventos_label = QtWidgets.QLabel("Eventos de Riesgo:")
        eventos_label.setAlignment(QtCore.Qt.AlignTop)
        eventos_layout.addWidget(eventos_label)

        self.eventos_table = QtWidgets.QTableWidget(0, 1)
        self.eventos_table.setHorizontalHeaderLabels(["Nombre del Evento"])
        self.eventos_table.horizontalHeader().setStretchLastSection(True)
        self.eventos_table.setSelectionBehavior(QtWidgets.QAbstractItemView.SelectRows)
        self.eventos_table.setSelectionMode(QtWidgets.QAbstractItemView.MultiSelection)
        eventos_layout.addWidget(self.eventos_table)
        layout.addLayout(eventos_layout)

        # Conectar la señal de doble clic a la función editar_evento_desde_tabla
        self.eventos_table.cellDoubleClicked.connect(self.editar_evento_desde_tabla)

        # Botones de acciones sobre eventos
        acciones_layout = QtWidgets.QHBoxLayout()
        editar_evento_button = QtWidgets.QPushButton("Editar Evento")
        editar_evento_button.setIcon(QtGui.QIcon.fromTheme("edit"))
        duplicar_evento_button = QtWidgets.QPushButton("Duplicar Evento(s)")
        duplicar_evento_button.setIcon(QtGui.QIcon.fromTheme("edit-copy"))
        eliminar_evento_button = QtWidgets.QPushButton("Eliminar Evento(s)")
        eliminar_evento_button.setIcon(QtGui.QIcon.fromTheme("edit-delete"))
        editar_evento_button.clicked.connect(self.editar_evento_popup)
        duplicar_evento_button.clicked.connect(self.duplicar_eventos)
        eliminar_evento_button.clicked.connect(self.eliminar_evento)
        acciones_layout.addWidget(editar_evento_button)
        acciones_layout.addWidget(duplicar_evento_button)
        acciones_layout.addWidget(eliminar_evento_button)
        acciones_layout.addStretch()
        layout.addLayout(acciones_layout)

        # Botón para ejecutar simulación
        simular_button = QtWidgets.QPushButton("Ejecutar Simulación")
        simular_button.setIcon(QtGui.QIcon.fromTheme("system-run"))
        simular_button.clicked.connect(self.ejecutar_simulacion)
        layout.addWidget(simular_button)

    def editar_evento_desde_tabla(self, row, column):
        self.editar_evento_popup(new=False, row=row)

    def setup_results_tab(self):
        layout = QtWidgets.QVBoxLayout(self.results_tab)

        # Barra de progreso
        self.progress_bar = QtWidgets.QProgressBar()
        self.progress_bar.setMaximum(100)
        self.progress_bar.setValue(0)
        layout.addWidget(self.progress_bar)

        # Texto de resultados
        self.resultados_text_edit = QtWidgets.QTextEdit()
        self.resultados_text_edit.setReadOnly(True)
        layout.addWidget(self.resultados_text_edit)

        # Pestañas para los gráficos
        self.graficos_tab_widget = QtWidgets.QTabWidget()
        layout.addWidget(self.graficos_tab_widget)

    def apply_stylesheet(self):
        """Aplica una hoja de estilo QSS a la aplicación."""
        style = """
        QWidget {
            font-family: Helvetica;
            font-size: 14px;
        }
        QMainWindow {
            background-color: #f0f0f0;
        }
        QPushButton {
            background-color: #007ACC;
            color: white;
            border-radius: 5px;
            padding: 5px;
        }
        QPushButton:hover {
            background-color: #005F8C;
        }
        QLineEdit, QComboBox {
            padding: 5px;
            border: 1px solid #ccc;
            border-radius: 3px;
        }
        QLabel {
            font-size: 14px;
        }
        QTabWidget::pane {
            border: 1px solid #ccc;
            background-color: #ffffff;
        }
        QTabBar::tab {
            background: #e0e0e0;
            border: 1px solid #ccc;
            padding: 8px;
        }
        QTabBar::tab:selected {
            background: #ffffff;
            border-bottom: 1px solid #ffffff;
        }
        QHeaderView::section {
            background-color: #f0f0f0;
            padding: 4px;
            border: none;
            border-bottom: 1px solid #ccc;
        }
        """
        self.setStyleSheet(style)

    def agregar_evento_popup(self):
        self.editar_evento_popup(new=True)

    def editar_evento_popup(self, new=False, row=None): # La firma no cambia

        evento_data = None # Inicializamos evento_data

        if not new:
            if row is not None:
                # Llamado por doble clic (o similar pasando 'row')
                # Usamos directamente la fila proporcionada
                if 0 <= row < len(self.eventos_riesgo): # Comprobación de seguridad
                    evento_data = self.eventos_riesgo[row]
                else:
                    QtWidgets.QMessageBox.critical(self, "Error", f"Índice de fila inválido: {row}")
                    return # Salir si la fila no es válida
            else:
                # Llamado por el botón "Editar Evento" (row es None)
                # Aquí SÍ verificamos la selección de la tabla
                selected_items = self.eventos_table.selectionModel().selectedRows()
                if not selected_items:
                    QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione un evento para editar.")
                    return
                if len(selected_items) > 1:
                    QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione solo UN evento para editar con el botón.")
                    return
                # Si llegamos aquí, hay exactamente una fila seleccionada
                row = selected_items[0].row() # Obtenemos la fila de la selección
                evento_data = self.eventos_riesgo[row]

            # Si por alguna razón no pudimos obtener evento_data (ej. fila inválida)
            if evento_data is None:
                print("Error: No se pudo obtener los datos del evento para editar.") # Mensaje consola
                return # Salir

        else: # new es True, esto no cambia
            evento_data = None # Para un evento nuevo, no hay datos previos

        # Crear diálogo
        dialog = QtWidgets.QDialog(self)
        dialog.setWindowTitle("Agregar Evento de Riesgo" if new else "Editar Evento de Riesgo")
        dialog_layout = QtWidgets.QVBoxLayout(dialog)

        # Nombre del Evento
        nombre_layout = QtWidgets.QHBoxLayout()
        nombre_label = QtWidgets.QLabel("Nombre del Evento:")
        nombre_layout.addWidget(nombre_label)
        nombre_var = QtWidgets.QLineEdit(evento_data['nombre'] if evento_data else "")
        nombre_layout.addWidget(nombre_var)
        dialog_layout.addLayout(nombre_layout)

        # Distribución de Severidad
        sev_group = QtWidgets.QGroupBox("Distribución de Impacto")
        sev_layout = QtWidgets.QVBoxLayout(sev_group)

        opciones_severidad = [("Normal", 1), ("LogNormal", 2), ("PERT", 3), ("Pareto", 4), ("Uniforme", 5)]
        distribuciones_severidad = [op[0] for op in opciones_severidad]
        distribucion_severidad_label = QtWidgets.QLabel("Seleccione la distribución de Impacto:")
        sev_layout.addWidget(distribucion_severidad_label)
        sev_combobox = QtWidgets.QComboBox()
        sev_combobox.addItems(distribuciones_severidad)
        if evento_data:
            sev_combobox.setCurrentIndex(evento_data['sev_opcion'] - 1)
        sev_layout.addWidget(sev_combobox)

        # --- NUEVO: Selector de Método de Entrada (Solo para Lognormal/GPD) ---
        sev_input_method_label = QtWidgets.QLabel("Método de Entrada de Parámetros:")
        sev_input_method_combo = QtWidgets.QComboBox()
        sev_input_method_combo.addItems(["Mínimo / Más Probable / Máximo", "Parámetros Directos"])
        # Leer el método guardado si estamos editando
        current_input_method_index = 0 # Por defecto: Min/Mode/Max
        if evento_data and evento_data.get('sev_input_method') == 'direct':
             current_input_method_index = 1
        sev_input_method_combo.setCurrentIndex(current_input_method_index)

        sev_layout.addWidget(sev_input_method_label)
        sev_layout.addWidget(sev_input_method_combo)
        # Inicialmente ocultos, se mostrarán si se elige Lognormal o GPD
        sev_input_method_label.hide()
        sev_input_method_combo.hide()

        # --- MODIFICADO: Usaremos un QStackedWidget para cambiar entre los grupos de parámetros ---
        sev_params_stack = QtWidgets.QStackedWidget()
        sev_layout.addWidget(sev_params_stack) # Añadimos el StackedWidget al layout principal de severidad

        # --- Grupo 0: Parámetros Min/Mode/Max  ---
        min_mode_max_widget = QtWidgets.QWidget()
        min_mode_max_layout = QtWidgets.QFormLayout(min_mode_max_widget)

        sev_min_var = QtWidgets.QLineEdit(str(evento_data.get('sev_minimo', '')) if evento_data else "")
        sev_mas_probable_var = QtWidgets.QLineEdit(str(evento_data.get('sev_mas_probable', '')) if evento_data else "")
        sev_max_var = QtWidgets.QLineEdit(str(evento_data.get('sev_maximo', '')) if evento_data else "")

        min_mode_max_layout.addRow("Valor Mínimo:", sev_min_var)
        sev_mas_probable_label = QtWidgets.QLabel("Valor Más Probable:") # Guardamos referencia para ocultar/mostrar
        min_mode_max_layout.addRow(sev_mas_probable_label, sev_mas_probable_var)
        min_mode_max_layout.addRow("Valor Máximo:", sev_max_var)
        sev_params_stack.addWidget(min_mode_max_widget) # Añadimos este widget al stack (índice 0)

        # --- Grupo 1: Parámetros Directos Lognormal ---
        direct_ln_widget = QtWidgets.QWidget()
        direct_ln_layout = QtWidgets.QFormLayout(direct_ln_widget)
        # Obtener valores guardados si existen
        ln_params = evento_data.get('sev_params_direct', {}) if evento_data and evento_data.get('sev_input_method') == 'direct' else {}

        sev_ln_s_var = QtWidgets.QLineEdit(str(ln_params.get('s', '')))
        sev_ln_scale_var = QtWidgets.QLineEdit(str(ln_params.get('scale', '')))
        sev_ln_loc_var = QtWidgets.QLineEdit(str(ln_params.get('loc', '0'))) # loc a menudo es 0

        direct_ln_layout.addRow("Shape (s o sigma):", sev_ln_s_var)
        direct_ln_layout.addRow("Scale (exp(mu)):", sev_ln_scale_var)
        direct_ln_layout.addRow("Location (loc, opcional):", sev_ln_loc_var)
        sev_params_stack.addWidget(direct_ln_widget) # Añadimos este widget al stack (índice 1)

        # --- Grupo 2: Parámetros Directos GPD ---
        direct_gpd_widget = QtWidgets.QWidget()
        direct_gpd_layout = QtWidgets.QFormLayout(direct_gpd_widget)
        # Obtener valores guardados si existen
        gpd_params = evento_data.get('sev_params_direct', {}) if evento_data and evento_data.get('sev_input_method') == 'direct' else {}

        sev_gpd_c_var = QtWidgets.QLineEdit(str(gpd_params.get('c', '')))
        sev_gpd_scale_var = QtWidgets.QLineEdit(str(gpd_params.get('scale', '')))
        sev_gpd_loc_var = QtWidgets.QLineEdit(str(gpd_params.get('loc', '')))

        direct_gpd_layout.addRow("Shape (c o xi):", sev_gpd_c_var)
        direct_gpd_layout.addRow("Scale (beta):", sev_gpd_scale_var)
        direct_gpd_layout.addRow("Location (loc o umbral):", sev_gpd_loc_var)
        sev_params_stack.addWidget(direct_gpd_widget) # Añadimos este widget al stack (índice 2)

        dialog_layout.addWidget(sev_group) # Añadimos el QGroupBox principal al diálogo

        # --- MODIFICADO: Función para actualizar visibilidad de parámetros de Severidad ---
        def actualizar_parametros_severidad():
            dist_seleccionada_index = sev_combobox.currentIndex()
            dist_nombre = opciones_severidad[dist_seleccionada_index][0] # "Normal", "LogNormal", etc.
            input_method_index = sev_input_method_combo.currentIndex() # 0: Min/Mode/Max, 1: Directo

            # Ocultar/Mostrar selector de método
            if dist_nombre in ["LogNormal", "Pareto"]: # Usamos "Pareto" como el nombre en la lista para GPD
                sev_input_method_label.show()
                sev_input_method_combo.show()
            else:
                sev_input_method_label.hide()
                sev_input_method_combo.hide()

            # Cambiar el widget visible en el StackedWidget
            if dist_nombre == "LogNormal" and input_method_index == 1: # LogNormal Directo
                sev_params_stack.setCurrentIndex(1)
            elif dist_nombre == "Pareto" and input_method_index == 1: # GPD Directo
                 sev_params_stack.setCurrentIndex(2)
            else: # Cualquier otra distribución o Lognormal/GPD con Min/Mode/Max
                sev_params_stack.setCurrentIndex(0)
                # Adicionalmente, ocultar 'Mas Probable' para Uniforme dentro del widget Min/Mode/Max
                if dist_nombre == "Uniforme":
                    sev_mas_probable_label.hide()
                    sev_mas_probable_var.hide()
                else:
                    sev_mas_probable_label.show()
                    sev_mas_probable_var.show()

        # Conectar señales para actualizar la UI cuando cambie la distribución o el método
        sev_combobox.currentIndexChanged.connect(actualizar_parametros_severidad)
        sev_input_method_combo.currentIndexChanged.connect(actualizar_parametros_severidad)
        # Llamada inicial para configurar la UI correctamente al abrir el diálogo
        actualizar_parametros_severidad()

        # Distribución de Frecuencia
        freq_group = QtWidgets.QGroupBox("Distribución de Frecuencia")
        freq_layout = QtWidgets.QVBoxLayout(freq_group)

        opciones_frecuencia = [("Poisson", 1), ("Binomial", 2), ("Bernoulli", 3)]
        distribuciones_frecuencia = [op[0] for op in opciones_frecuencia]
        distribucion_frecuencia_label = QtWidgets.QLabel("Seleccione la distribución de frecuencia:")
        freq_layout.addWidget(distribucion_frecuencia_label)
        freq_combobox = QtWidgets.QComboBox()
        freq_combobox.addItems(distribuciones_frecuencia)
        if evento_data:
            freq_combobox.setCurrentIndex(evento_data['freq_opcion'] - 1)
        freq_layout.addWidget(freq_combobox)

        # Parámetros de Frecuencia
        freq_params_stack = QtWidgets.QStackedWidget()

        # Parámetros Poisson
        poisson_widget = QtWidgets.QWidget()
        poisson_layout = QtWidgets.QFormLayout(poisson_widget)
        tasa_var = QtWidgets.QLineEdit(str(evento_data['tasa']) if evento_data and 'tasa' in evento_data else "")
        poisson_layout.addRow("Tasa Media (λ):", tasa_var)
        freq_params_stack.addWidget(poisson_widget)

        # Parámetros Binomial
        binomial_widget = QtWidgets.QWidget()
        binomial_layout = QtWidgets.QFormLayout(binomial_widget)
        num_eventos_var = QtWidgets.QLineEdit(str(evento_data['num_eventos']) if evento_data and 'num_eventos' in evento_data else "")
        prob_exito_var = QtWidgets.QLineEdit(str(evento_data['prob_exito']) if evento_data and 'prob_exito' in evento_data else "")
        binomial_layout.addRow("Número de Eventos Posibles (n):", num_eventos_var)
        binomial_layout.addRow("Probabilidad de Éxito (p):", prob_exito_var)
        freq_params_stack.addWidget(binomial_widget)

        # Parámetros Bernoulli
        bernoulli_widget = QtWidgets.QWidget()
        bernoulli_layout = QtWidgets.QFormLayout(bernoulli_widget)
        prob_exito_var_bern = QtWidgets.QLineEdit(str(evento_data['prob_exito']) if evento_data and 'prob_exito' in evento_data else "")
        bernoulli_layout.addRow("Probabilidad de Éxito (p):", prob_exito_var_bern)
        freq_params_stack.addWidget(bernoulli_widget)

        freq_layout.addWidget(freq_params_stack)
        dialog_layout.addWidget(freq_group)

        # Función para actualizar parámetros de Frecuencia
        def actualizar_parametros_frecuencia():
            opcion = freq_combobox.currentIndex() + 1
            if opcion == 1:
                freq_params_stack.setCurrentIndex(0)
            elif opcion == 2:
                freq_params_stack.setCurrentIndex(1)
            elif opcion == 3:
                freq_params_stack.setCurrentIndex(2)
        freq_combobox.currentIndexChanged.connect(actualizar_parametros_frecuencia)
        actualizar_parametros_frecuencia()

        # Eventos y Tipos de Vínculo
        vinculos_label = QtWidgets.QLabel("Vínculos con otros eventos:")
        dialog_layout.addWidget(vinculos_label)

        # Tabla para gestionar vínculos
        vinculos_table = QtWidgets.QTableWidget()
        vinculos_table.setColumnCount(3)
        vinculos_table.setHorizontalHeaderLabels(["Evento", "Tipo de Vínculo", "Eliminar"])
        vinculos_table.horizontalHeader().setStretchLastSection(False)
        vinculos_table.setColumnWidth(0, 200)  # Ajusta este valor según necesites
        vinculos_table.setColumnWidth(1, 100)
        vinculos_table.setColumnWidth(2, 80)
        dialog_layout.addWidget(vinculos_table)

        # Botón para añadir nuevo vínculo
        add_vinculo_btn = QtWidgets.QPushButton("Añadir Vínculo")
        dialog_layout.addWidget(add_vinculo_btn)

        # Guardar los vínculos existentes
        vinculos_existentes = []
        if evento_data and 'vinculos' in evento_data:
            vinculos_existentes = evento_data['vinculos']
        elif evento_data and 'eventos_padres' in evento_data:
            # Convertir formato antiguo al nuevo
            tipo = evento_data.get('tipo_dependencia', 'AND')
            for padre_id in evento_data['eventos_padres']:
                vinculos_existentes.append({'id_padre': padre_id, 'tipo': tipo})

        # Función para añadir un vínculo a la tabla
        def añadir_vinculo():
            # Crear diálogo para seleccionar evento y tipo
            sel_dialog = QtWidgets.QDialog(dialog)
            sel_dialog.setWindowTitle("Añadir Vínculo")
            sel_dialog.setMinimumWidth(300)
            sel_layout = QtWidgets.QVBoxLayout(sel_dialog)

            # Lista de eventos disponibles (excluir el actual y los ya vinculados)
            eventos_disponibles = []
            vinculos_ids = [v['id_padre'] for v in vinculos_existentes]
            for e in self.eventos_riesgo:
                if (new or e != evento_data) and e['id'] not in vinculos_ids:
                    eventos_disponibles.append(e)

            if not eventos_disponibles:
                QtWidgets.QMessageBox.warning(sel_dialog, "Advertencia",
                                              "No hay eventos disponibles para vincular.")
                return

            # Selección de evento
            sel_layout.addWidget(QtWidgets.QLabel("Seleccionar evento:"))
            evento_combo = QtWidgets.QComboBox()
            for e in eventos_disponibles:
                evento_combo.addItem(e['nombre'], e['id'])
            sel_layout.addWidget(evento_combo)

            # Selección de tipo de vínculo
            sel_layout.addWidget(QtWidgets.QLabel("Tipo de vínculo:"))
            tipo_combo = QtWidgets.QComboBox()
            tipo_combo.addItems(["AND", "OR", "EXCLUYE"])
            sel_layout.addWidget(tipo_combo)

            # Explicación de los tipos de vínculo
            explicacion = QtWidgets.QLabel(
                "<b>AND</b>: Este evento ocurre solo si el evento vinculado también ocurre.\n"
                "<b>OR</b>: Este evento ocurre si al menos uno de los eventos vinculados con OR ocurre.\n"
                "<b>EXCLUYE</b>: Este evento ocurre solo si el evento vinculado NO ocurre."
            )
            explicacion.setWordWrap(True)
            sel_layout.addWidget(explicacion)

            # Botones de aceptar/cancelar
            buttons = QtWidgets.QDialogButtonBox(QtWidgets.QDialogButtonBox.Ok |
                                                QtWidgets.QDialogButtonBox.Cancel)
            buttons.accepted.connect(sel_dialog.accept)
            buttons.rejected.connect(sel_dialog.reject)
            sel_layout.addWidget(buttons)

            # Mostrar diálogo y procesar resultado
            if sel_dialog.exec_() == QtWidgets.QDialog.Accepted:
                evento_id = evento_combo.currentData()
                tipo = tipo_combo.currentText()

                # Agregar a la lista de vínculos existentes
                vinculos_existentes.append({'id_padre': evento_id, 'tipo': tipo})

                # Actualizar la tabla
                actualizar_tabla_vinculos()

        # Función para actualizar la tabla de vínculos
        def actualizar_tabla_vinculos():
            # Limpiar tabla
            vinculos_table.setRowCount(0)

            # Agregar vínculos existentes
            for idx, vinculo in enumerate(vinculos_existentes):
                vinculos_table.insertRow(idx)

                # Nombre del evento padre
                nombre_padre = "Desconocido"
                for e in self.eventos_riesgo:
                    if e['id'] == vinculo['id_padre']:
                        nombre_padre = e['nombre']
                        break

                # Celda con nombre del evento
                item_evento = QtWidgets.QTableWidgetItem(nombre_padre)
                item_evento.setFlags(item_evento.flags() & ~QtCore.Qt.ItemIsEditable)
                vinculos_table.setItem(idx, 0, item_evento)

                # Celda con combo para tipo de vínculo
                tipo_combo = QtWidgets.QComboBox()
                tipo_combo.addItems(["AND", "OR", "EXCLUYE"])
                tipo_combo.setCurrentText(vinculo['tipo'])
                tipo_combo.currentTextChanged.connect(lambda text, row=idx: actualizar_tipo_vinculo(row, text))
                vinculos_table.setCellWidget(idx, 1, tipo_combo)

                # Botón para eliminar
                delete_btn = QtWidgets.QPushButton("Eliminar")
                delete_btn.clicked.connect(lambda _, row=idx: eliminar_vinculo(row))
                vinculos_table.setCellWidget(idx, 2, delete_btn)

        # Función para actualizar el tipo de un vínculo
        def actualizar_tipo_vinculo(row, nuevo_tipo):
            vinculos_existentes[row]['tipo'] = nuevo_tipo

        # Función para eliminar un vínculo
        def eliminar_vinculo(row):
            del vinculos_existentes[row]
            actualizar_tabla_vinculos()

        # Conectar botón para añadir vínculo
        add_vinculo_btn.clicked.connect(añadir_vinculo)

        # Inicializar la tabla
        actualizar_tabla_vinculos()

        # Botones
        buttons = QtWidgets.QDialogButtonBox()
        buttons.setStandardButtons(QtWidgets.QDialogButtonBox.Save | QtWidgets.QDialogButtonBox.Cancel)
        dialog_layout.addWidget(buttons)

        buttons.accepted.connect(lambda: self.guardar_evento(
            dialog, new, row, nombre_var,
            sev_combobox, sev_input_method_combo, # Pasamos el combo de método
            sev_min_var, sev_mas_probable_var, sev_max_var, # Params Min/Mode/Max
            sev_ln_s_var, sev_ln_scale_var, sev_ln_loc_var, # Params Directos LN
            sev_gpd_c_var, sev_gpd_scale_var, sev_gpd_loc_var, # Params Directos GPD
            freq_combobox, tasa_var, num_eventos_var, prob_exito_var,
            prob_exito_var_bern, vinculos_existentes))

        buttons.rejected.connect(dialog.reject)

        dialog.exec_()

    # --- MODIFICADO: Añadir los nuevos parámetros a la firma de la función ---
    def guardar_evento(self, dialog, new, row, nombre_var,
                     sev_combobox, sev_input_method_combo, # <- Añadido combo método
                     sev_min_var, sev_mas_probable_var, sev_max_var,
                     sev_ln_s_var, sev_ln_scale_var, sev_ln_loc_var, # <- Añadidos params LN
                     sev_gpd_c_var, sev_gpd_scale_var, sev_gpd_loc_var, # <- Añadidos params GPD
                     freq_combobox, tasa_var, num_eventos_var,
                     prob_exito_var, prob_exito_var_bern, vinculos_existentes):
        try:
            nombre_evento = nombre_var.text().strip()
            if not nombre_evento:
                raise ValueError("El nombre del evento no puede estar vacío.")
            if len(nombre_evento) > 50:
                raise ValueError("El nombre del evento no puede tener más de 50 caracteres.")

            # Severidad
            # --- NUEVA LÓGICA PARA SEVERIDAD ---
            sev_opcion = sev_combobox.currentIndex() + 1
            dist_nombre_sev = sev_combobox.currentText() # "Normal", "LogNormal", etc.

            # Determinar el método de entrada
            sev_input_method = 'min_mode_max' # Por defecto
            if dist_nombre_sev in ["LogNormal", "Pareto"] and sev_input_method_combo.currentIndex() == 1:
                sev_input_method = 'direct'

            # Inicializar variables para guardar
            sev_minimo = None
            sev_mas_probable = None
            sev_maximo = None
            sev_params_direct = {} # Diccionario para parámetros directos

            dist_sev = None # Inicializamos la distribución

            if sev_input_method == 'direct':
                # --- Leer y Validar Parámetros Directos ---
                if dist_nombre_sev == "LogNormal":
                    try:
                        s = float(sev_ln_s_var.text())
                        scale = float(sev_ln_scale_var.text())
                        loc = float(sev_ln_loc_var.text() if sev_ln_loc_var.text() else '0') # Opcional, default 0
                        if s <= 0: raise ValueError("Shape (s) debe ser positivo para Lognormal.")
                        if scale <= 0: raise ValueError("Scale debe ser positivo para Lognormal.")
                        sev_params_direct = {'s': s, 'scale': scale, 'loc': loc}
                        # Intentar crear la distribución para validar parámetros
                        dist_sev = lognorm(s=s, scale=scale, loc=loc)
                    except ValueError as e:
                        raise ValueError(f"Error en parámetros Lognormal directos: {e}")
                elif dist_nombre_sev == "Pareto": # GPD
                    try:
                        c = float(sev_gpd_c_var.text())
                        scale = float(sev_gpd_scale_var.text())
                        loc = float(sev_gpd_loc_var.text())
                        if scale <= 0: raise ValueError("Scale (beta) debe ser positivo para GPD.")
                        # Podrían necesitarse más validaciones según el valor de 'c' y 'loc'
                        sev_params_direct = {'c': c, 'scale': scale, 'loc': loc}
                         # Intentar crear la distribución para validar parámetros
                        dist_sev = genpareto(c=c, scale=scale, loc=loc)
                    except ValueError as e:
                        raise ValueError(f"Error en parámetros GPD directos: {e}")

            else: # sev_input_method == 'min_mode_max'
                # --- Leer y Validar Parámetros Min/Mode/Max (como antes) ---
                try:
                    if not sev_min_var.text(): raise ValueError("Mínimo vacío.")
                    sev_minimo = float(sev_min_var.text())
                    if not sev_max_var.text(): raise ValueError("Máximo vacío.")
                    sev_maximo = float(sev_max_var.text())

                    # Más probable solo necesario si no es Uniforme
                    if dist_nombre_sev != "Uniforme":
                        if not sev_mas_probable_var.text(): raise ValueError("Más probable vacío.")
                        sev_mas_probable = float(sev_mas_probable_var.text())
                    else:
                        sev_mas_probable = None # No se usa para Uniforme

                    # Generar distribución usando el método antiguo (que llama a obtener_parametros_...)
                    # La función generar_distribucion_severidad ya valida Min <= Mode <= Max etc.
                    dist_sev = generar_distribucion_severidad(sev_opcion, sev_minimo, sev_mas_probable, sev_maximo)

                except ValueError as e:
                     raise ValueError(f"Error en parámetros Min/Mode/Max: {e}")

            # Si dist_sev no se pudo crear por alguna razón (debería haber lanzado error antes)
            if dist_sev is None:
                raise ValueError("No se pudo crear la distribución de severidad.")

            # Frecuencia
            freq_opcion = freq_combobox.currentIndex() + 1
            tasa = None
            num_eventos = None
            prob_exito = None
            dist_freq = None

            if freq_opcion == 1: # Poisson
                # ... leer tasa, validar, crear dist_freq ...
                 if not tasa_var.text(): raise ValueError("La tasa media (λ) no puede estar vacía.")
                 tasa = float(tasa_var.text())
                 if tasa <= 0: raise ValueError("La tasa media (λ) debe ser mayor que cero.")
                 dist_freq = generar_distribucion_frecuencia(freq_opcion, tasa=tasa)
            elif freq_opcion == 2: # Binomial
                # ... leer num_eventos, prob_exito, validar, crear dist_freq ...
                if not num_eventos_var.text(): raise ValueError("El número de eventos posibles (n) no puede estar vacío.")
                if not prob_exito_var.text(): raise ValueError("La probabilidad de éxito (p) no puede estar vacía.")
                num_eventos = int(num_eventos_var.text())
                prob_exito = float(prob_exito_var.text())
                if num_eventos <= 0: raise ValueError("El número de eventos posibles (n) debe ser mayor que cero.")
                if not 0 <= prob_exito <= 1: raise ValueError("La probabilidad de éxito (p) debe estar entre 0 y 1.")
                dist_freq = generar_distribucion_frecuencia(freq_opcion, num_eventos_posibles=num_eventos, probabilidad_exito=prob_exito)
            elif freq_opcion == 3: # Bernoulli
                # ... leer prob_exito_var_bern, validar, crear dist_freq ...
                if not prob_exito_var_bern.text(): raise ValueError("La probabilidad de éxito (p) no puede estar vacía.")
                prob_exito = float(prob_exito_var_bern.text())
                if not 0 <= prob_exito <= 1: raise ValueError("La probabilidad de éxito (p) debe estar entre 0 y 1.")
                dist_freq = generar_distribucion_frecuencia(freq_opcion, probabilidad_exito=prob_exito)

            if dist_freq is None:
                 raise ValueError("No se pudo crear la distribución de frecuencia.")

            # Obtener los vínculos de la tabla
            vinculos = vinculos_existentes.copy()

            # Verificar que vinculos_existentes es una lista válida
            if vinculos_existentes is None:
                vinculos = []
            else:
                # Hacer una copia profunda para evitar referencias
                vinculos = copy.deepcopy(vinculos_existentes)

            # Crear el evento temporal
            evento_temp = {
                # Resto de propiedades...
                'vinculos': vinculos
            }

            # Crear el evento temporal
            evento_temp = {
                'nombre': nombre_evento,
                'id': str(uuid.uuid4()) if new else self.eventos_riesgo[row]['id'], # Reusar ID si se edita

                # Campos de Severidad
                'sev_opcion': sev_opcion,
                'sev_input_method': sev_input_method, # <- NUEVO
                'sev_minimo': sev_minimo, # Puede ser None si es 'direct'
                'sev_mas_probable': sev_mas_probable, # Puede ser None si es 'direct' o 'Uniforme'
                'sev_maximo': sev_maximo, # Puede ser None si es 'direct'
                'sev_params_direct': sev_params_direct, # <- NUEVO (diccionario vacío si es 'min_mode_max')
                'dist_severidad': dist_sev, # Objeto distribución creado

                # Campos de Frecuencia (sin cambios en estructura)
                'freq_opcion': freq_opcion,
                'tasa': tasa,
                'num_eventos': num_eventos,
                'prob_exito': prob_exito,
                'dist_frecuencia': dist_freq, # Objeto distribución creado

                # Vínculos
                'vinculos': vinculos_existentes.copy() # Asegúrate que esto funcione como antes
            }

            # Asignar un ID al evento
            if not new:
                # Obtener el ID existente
                evento_temp['id'] = self.eventos_riesgo[row]['id']
            else:
                # Asignar un nuevo ID único
                evento_temp['id'] = str(uuid.uuid4())

            # Validar que no se formen ciclos en las dependencias
            eventos_temp = self.eventos_riesgo.copy()
            if new:
                eventos_temp.append(evento_temp)
            else:
                eventos_temp[row] = evento_temp

            if self.tiene_ciclo(eventos_temp):
                raise ValueError("Agregar estos eventos vinculados genera un error de dependencia cíclica.")

            # Guardar evento
            if new:
                self.eventos_riesgo.append(evento_temp)
                row_position = self.eventos_table.rowCount()
                self.eventos_table.insertRow(row_position)
                self.eventos_table.setItem(row_position, 0, QtWidgets.QTableWidgetItem(nombre_evento))
            else:
                self.eventos_riesgo[row] = evento_temp
                self.eventos_table.setItem(row, 0, QtWidgets.QTableWidgetItem(nombre_evento))

            dialog.accept()

        except ValueError as ve:
            QtWidgets.QMessageBox.critical(dialog, "Error de Validación", str(ve)) # Usar 'dialog' como parent
        except Exception as e:
            error_message = traceback.format_exc()
            QtWidgets.QMessageBox.critical(dialog, "Error Inesperado", f"No se pudo guardar el evento:\n{error_message}") # Usar 'dialog'

    def tiene_ciclo(self, eventos_riesgo):
        id_a_evento = {evento['id']: evento for evento in eventos_riesgo}
        visited = set()
        rec_stack = set()

        def is_cyclic_util(evento_id):
            visited.add(evento_id)
            rec_stack.add(evento_id)
            evento = id_a_evento[evento_id]

            # Obtener IDs de padres desde la nueva estructura
            padres_ids = []
            if 'vinculos' in evento:
                padres_ids = [vinculo['id_padre'] for vinculo in evento['vinculos']]
            elif 'eventos_padres' in evento:
                # Compatibilidad con formato antiguo
                padres_ids = evento.get('eventos_padres', [])

            for padre_id in padres_ids:
                if padre_id not in visited:
                    if is_cyclic_util(padre_id):
                        return True
                elif padre_id in rec_stack:
                    return True
            rec_stack.remove(evento_id)
            return False

        for evento in eventos_riesgo:
            if evento['id'] not in visited:
                if is_cyclic_util(evento['id']):
                    return True
        return False

    def eliminar_evento(self):
        selected_items = self.eventos_table.selectionModel().selectedRows()
        if not selected_items:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione al menos un evento para eliminar.")
            return

        # Confirmar eliminación múltiple
        respuesta = QtWidgets.QMessageBox.question(
            self,
            "Eliminar Eventos",
            f"¿Estás seguro de que deseas eliminar {len(selected_items)} evento(s)?",
            QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No
        )

        if respuesta == QtWidgets.QMessageBox.Yes:
            # Ordenar los índices de fila en orden descendente para evitar problemas al eliminar filas
            rows = sorted([index.row() for index in selected_items], reverse=True)
            for row in rows:
                del self.eventos_riesgo[row]
                self.eventos_table.removeRow(row)

    def duplicar_eventos(self):
        selected_items = self.eventos_table.selectionModel().selectedRows()
        if not selected_items:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione al menos un evento para duplicar.")
            return

        # Mapeo entre IDs originales y nuevos IDs para los eventos duplicados
        id_original_a_nuevo = {}
        # Lista para almacenar los nuevos eventos duplicados
        nuevos_eventos = []
        eventos_a_duplicar = []

        for index in selected_items:
            row = index.row()
            evento_original = self.eventos_riesgo[row]
            eventos_a_duplicar.append(evento_original)

        # Primero asignamos nuevos IDs
        for evento_original in eventos_a_duplicar:
            evento_nuevo = evento_original.copy()
            nuevo_id = str(uuid.uuid4())
            id_original_a_nuevo[evento_original['id']] = nuevo_id
            evento_nuevo['id'] = nuevo_id
            # Añadir un prefijo o sufijo al nombre para indicar que es una copia
            evento_nuevo['nombre'] = evento_nuevo['nombre'] + " (Copia)"
            nuevos_eventos.append(evento_nuevo)

        # Actualizar las dependencias de los eventos duplicados
        for evento_nuevo in nuevos_eventos:
            # Manejar la nueva estructura de vínculos
            if 'vinculos' in evento_nuevo:
                vinculos_actualizados = []
                for vinculo in evento_nuevo.get('vinculos', []):
                    padre_id = vinculo['id_padre']
                    tipo = vinculo['tipo']
                    # Si el evento padre fue duplicado, usamos el nuevo ID
                    if padre_id in id_original_a_nuevo:
                        vinculos_actualizados.append({
                            'id_padre': id_original_a_nuevo[padre_id],
                            'tipo': tipo
                        })
                    else:
                        # Si no, mantenemos el ID original
                        vinculos_actualizados.append({
                            'id_padre': padre_id,
                            'tipo': tipo
                        })
                evento_nuevo['vinculos'] = vinculos_actualizados

            # Compatibilidad con formato antiguo
            elif 'eventos_padres' in evento_nuevo:
                eventos_padres_originales = evento_nuevo.get('eventos_padres', [])
                eventos_padres_actualizados = []
                for padre_id in eventos_padres_originales:
                    # Si el evento padre fue duplicado, usamos el nuevo ID
                    if padre_id in id_original_a_nuevo:
                        eventos_padres_actualizados.append(id_original_a_nuevo[padre_id])
                    else:
                        # Si no, mantenemos el ID original
                        eventos_padres_actualizados.append(padre_id)
                evento_nuevo['eventos_padres'] = eventos_padres_actualizados

        # Añadir los nuevos eventos a la lista y a la tabla
        for evento_nuevo in nuevos_eventos:
            self.eventos_riesgo.append(evento_nuevo)
            row_position = self.eventos_table.rowCount()
            self.eventos_table.insertRow(row_position)
            self.eventos_table.setItem(row_position, 0, QtWidgets.QTableWidgetItem(evento_nuevo['nombre']))

        QtWidgets.QMessageBox.information(self, "Duplicar Evento(s)", f"Se han duplicado {len(nuevos_eventos)} evento(s).")

    def mostrar_resultados_en_interfaz(self, texto):
        self.resultados_text_edit.clear()
        self.resultados_text_edit.setPlainText(texto)

    def solicitar_ruta_guardado_pdf(self):
        options = QtWidgets.QFileDialog.Options()
        filepath, _ = QtWidgets.QFileDialog.getSaveFileName(self, "Guardar Informe PDF", "",
                                                            "PDF Files (*.pdf);;All Files (*)", options=options)
        return filepath

    def ejecutar_simulacion(self):
        try:
            num_simulaciones_text = self.num_simulaciones_var.text()
            if not num_simulaciones_text:
                raise ValueError("El número de simulaciones no puede estar vacío.")
            try:
                num_simulaciones = int(num_simulaciones_text)
            except ValueError:
                raise ValueError("El número de simulaciones debe ser un entero válido.")
            if num_simulaciones < 1000:
                raise ValueError("El número de simulaciones debe ser al menos 1000.")
            if not self.eventos_riesgo:
                raise ValueError("Debe agregar al menos un evento de riesgo.")

            # Se determina qué eventos usar, dependiendo si hay un escenario seleccionado
            if self.current_scenario:
                eventos = self.current_scenario.eventos_riesgo
            else:
                eventos = self.eventos_riesgo

            # Por defecto, no generamos reporte automáticamente
            self.generar_reporte = False
            self.pdf_filename = 'reporte_simulacion.pdf'  # Valor por defecto, no se usará

            # Preparar la lista de eventos para la simulación utilizando la variable 'eventos'
            eventos_simulacion = []
            for evento_data in eventos:
                evento = {
                    'id': evento_data['id'],
                    'nombre': evento_data['nombre'],
                    'dist_severidad': evento_data['dist_severidad'],
                    'dist_frecuencia': evento_data['dist_frecuencia']
                }

                # Incluir vínculos si existen
                if 'vinculos' in evento_data:
                    evento['vinculos'] = evento_data['vinculos']

                # Para compatibilidad con formato antiguo
                elif 'eventos_padres' in evento_data:
                    evento['eventos_padres'] = evento_data['eventos_padres']
                    evento['tipo_dependencia'] = evento_data.get('tipo_dependencia', 'AND')

                eventos_simulacion.append(evento)

            # Desactivar la interfaz mientras se ejecuta la simulación
            self.set_interfaz_activa(False)
            self.progress_bar.setValue(0)
            self.central_widget.setCurrentWidget(self.results_tab)

            # Crear y configurar el hilo de simulación usando los eventos preparados
            self.simulation_thread = SimulacionThread(
                eventos_simulacion,
                num_simulaciones,
                self.generar_reporte,
                self.pdf_filename
            )
            self.simulation_thread.progreso_actualizado.connect(self.actualizar_progreso)
            self.simulation_thread.simulacion_completada.connect(self.simulacion_completada)
            self.simulation_thread.error_ocurrido.connect(self.simulacion_error)
            self.simulation_thread.start()

        except ValueError as ve:
            QtWidgets.QMessageBox.critical(self, "Error", str(ve))
        except Exception as e:
            QtWidgets.QMessageBox.critical(self, "Error", f"Error al ejecutar la simulación: {e}")

    def actualizar_progreso(self, valor):
        self.progress_bar.setValue(valor)

    def simulacion_completada(self, perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento, eventos):
        # Reactivar la interfaz
        self.set_interfaz_activa(True)
        self.progress_bar.setValue(100)

        # Guardar los resultados para uso posterior (exportación a PDF)
        self.resultados_simulacion = {
            'perdidas_totales': perdidas_totales,
            'frecuencias_totales': frecuencias_totales,
            'perdidas_por_evento': perdidas_por_evento,
            'frecuencias_por_evento': frecuencias_por_evento,
            'eventos_riesgo': eventos
        }


        # Obtener texto de resultados
        resultados_texto = self.generar_resultados(
            perdidas_totales,
            frecuencias_totales,
            perdidas_por_evento,
            frecuencias_por_evento,
            eventos,
            generar_reporte=self.generar_reporte,
            pdf_filename=self.pdf_filename
        )

        # Mostrar resultados en la interfaz
        self.mostrar_resultados_en_interfaz(resultados_texto)

        # Graficar resultados en la pestaña de Resultados
        self.graficar_resultados(perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento, eventos)

        # Cambiar a la pestaña de Resultados
        self.central_widget.setCurrentWidget(self.results_tab)

    def simulacion_error(self, mensaje_error):
        # Reactivar la interfaz
        self.set_interfaz_activa(True)
        QtWidgets.QMessageBox.critical(self, "Error", f"Error al ejecutar la simulación: {mensaje_error}")

    def set_interfaz_activa(self, estado):
        self.num_simulaciones_var.setEnabled(estado)
        self.eventos_table.setEnabled(estado)
        self.central_widget.setTabEnabled(self.central_widget.indexOf(self.config_tab), estado)

    def generar_resultados(self, perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento,
                           eventos_riesgo, generar_reporte=False, pdf_filename='reporte_simulacion.pdf'):
        """Calcula estadísticas, muestra resultados y genera gráficos."""
        texto_resultados = ""

        # Calculamos estadísticas para pérdidas agregadas
        percentiles_valores = [10, 20, 30, 40, 50, 60, 70, 75, 80, 85, 90, 92, 95, 99, 99.99]
        percentiles = np.percentile(perdidas_totales, percentiles_valores)
        media = np.mean(perdidas_totales)
        desviacion_estandar = np.std(perdidas_totales)

        # Estadísticas de frecuencias agregadas
        min_freq_total = int(frecuencias_totales.min())
        max_freq_total = int(frecuencias_totales.max())
        mode_freq_total = int(stats.mode(frecuencias_totales, keepdims=True).mode[0]) if len(frecuencias_totales) > 0 else 0

        # Creamos DataFrame de percentiles para pérdidas agregadas
        percentiles_df = pd.DataFrame({
            'Percentil (%)': percentiles_valores,
            'Valor de Pérdida': percentiles
        })

        # Formateamos los valores de pérdida sin decimales
        percentiles_df['Valor de Pérdida'] = percentiles_df['Valor de Pérdida'].round(0).astype(int)
        percentiles_df['Valor de Pérdida'] = percentiles_df['Valor de Pérdida'].apply(currency_format)

        # Formateamos los percentiles (%), dejando decimales donde corresponda
        percentiles_df['Percentil (%)'] = percentiles_df['Percentil (%)'].apply(
            lambda x: ('{:.2f}'.format(x).replace('.', ',')) if x != int(x) else ('{:.0f}'.format(x)))

        # Obtener resumen ejecutivo
        var_90 = np.percentile(perdidas_totales, 90)
        opvar_99 = np.percentile(perdidas_totales, 99)
        opvar = perdidas_totales[perdidas_totales >= opvar_99].mean()
        texto_resultados += obtener_resumen_ejecutivo_texto(media, desviacion_estandar, var_90, opvar_99, opvar,
                                                            min_freq_total, mode_freq_total, max_freq_total)

        # Obtener tabla de percentiles
        texto_resultados += obtener_tabla_percentiles_texto(percentiles_df, "Percentiles de Pérdida Agregada")

        # Creamos una lista para almacenar las estadísticas de cada evento para la matriz de riesgos
        estadisticas_eventos = []

        # Agregamos cálculo y muestra de estadísticas para cada evento de riesgo
        for idx, (perdidas_evento, frecuencias_evento) in enumerate(zip(perdidas_por_evento, frecuencias_por_evento)):
            nombre_evento = eventos_riesgo[idx]['nombre']
            media_evento = np.mean(perdidas_evento)
            desviacion_evento = np.std(perdidas_evento)
            percentiles_evento = np.percentile(perdidas_evento, percentiles_valores)
            min_freq = int(frecuencias_evento.min())
            max_freq = int(frecuencias_evento.max())
            mode_freq = int(stats.mode(frecuencias_evento, keepdims=True).mode[0]) if len(frecuencias_evento) > 0 else 0
            percentiles_evento_df = pd.DataFrame({
                'Percentil (%)': percentiles_valores,
                'Valor de Pérdida': percentiles_evento
            })
            percentiles_evento_df['Valor de Pérdida'] = percentiles_evento_df['Valor de Pérdida'].round(0).astype(int)
            percentiles_evento_df['Valor de Pérdida'] = percentiles_evento_df['Valor de Pérdida'].apply(currency_format)
            percentiles_evento_df['Percentil (%)'] = percentiles_evento_df['Percentil (%)'].apply(
                lambda x: ('{:.2f}'.format(x).replace('.', ',')) if x != int(x) else ('{:.0f}'.format(x)))

            # Obtenemos el percentil 90 del impacto para el evento
            p90_evento = np.percentile(perdidas_evento, 90)

            # Guardamos las estadísticas en la lista
            estadisticas_eventos.append({
                'nombre': nombre_evento,
                'impacto_medio': media_evento,
                'impacto_p90': p90_evento,
                'frecuencia_modo': mode_freq,
                'frecuencia_maxima': max_freq
            })

            texto_resultados += f"\nEstadísticas para el Evento de Riesgo: {nombre_evento}\n"
            texto_resultados += f"Media de Impacto: {currency_format(round(media_evento))}\n"
            texto_resultados += f"Desviación Estándar: {currency_format(round(desviacion_evento))}\n"
            texto_resultados += f"Número mínimo de eventos materializados: {min_freq}\n"
            texto_resultados += f"Número más probable de eventos materializados: {mode_freq}\n"
            texto_resultados += f"Número máximo de eventos materializados: {max_freq}\n"
            texto_resultados += "Percentiles de Pérdida:\n"
            texto_resultados += tabulate(percentiles_evento_df, headers='keys', tablefmt='fancy_grid', showindex=False)
            texto_resultados += "\n"

        if generar_reporte:
            try:
                # Crear el PDF usando ReportLab
                c = canvas.Canvas(pdf_filename, pagesize=A4)
                ancho_pagina, alto_pagina = A4
                # Establecer un margen
                margen = 2 * cm
                ancho_texto = ancho_pagina - 2 * margen
                alto_texto = alto_pagina - 2 * margen
                # Crear un objeto de texto
                texto_page = c.beginText(margen, alto_pagina - margen)
                texto_page.setFont("Helvetica", 10)

                # Dividir el texto en líneas y agregarlo al PDF
                lineas = texto_resultados.split('\n')
                for linea in lineas:
                    texto_page.textLine(linea)
                    if texto_page.getY() < margen:
                        # Añadir la página de texto y comenzar una nueva
                        c.drawText(texto_page)
                        c.showPage()
                        texto_page = c.beginText(margen, alto_pagina - margen)
                        texto_page.setFont("Helvetica", 10)
                c.drawText(texto_page)
                c.showPage()

                # Generar los gráficos y agregarlos al PDF
                figuras = self.generar_figuras(perdidas_totales, frecuencias_totales, perdidas_por_evento,
                                               frecuencias_por_evento, eventos_riesgo)

                for idx, figura in enumerate(figuras):
                    # Guardar la figura en memoria en lugar de en un archivo
                    img_data = io.BytesIO()
                    figura.savefig(img_data, format='png', dpi=300, bbox_inches='tight')
                    img_data.seek(0)  # Restablece el cursor al inicio del archivo en memoria

                    # Obtener el tamaño de la imagen
                    img_width, img_height = figura.get_size_inches()
                    img_width *= 300  # Convertir tamaño a píxeles (dpi)
                    img_height *= 300

                    # Calcular el escalado para ajustar la imagen al tamaño de la página
                    ratio = min((ancho_texto) / img_width, (alto_texto) / img_height)
                    scaled_width = img_width * ratio
                    scaled_height = img_height * ratio

                    c.drawImage(
                        ImageReader(img_data),
                        margen,
                        (alto_pagina - margen - scaled_height),
                        width=scaled_width,
                        height=scaled_height,
                        preserveAspectRatio=True,
                        anchor='c'
                    )
                    c.showPage()  # Añadir una nueva página para la siguiente imagen

                c.save()

                QtWidgets.QMessageBox.information(self, "Reporte PDF", f"El reporte ha sido guardado exitosamente en {pdf_filename}.")
            except Exception as e:
                error_message = traceback.format_exc()
                QtWidgets.QMessageBox.critical(self, "Error", f"No se pudo generar el reporte PDF:\n{error_message}")

        return texto_resultados

    def setup_scenarios_tab(self):
        layout = QtWidgets.QVBoxLayout(self.scenarios_tab)

        # Título
        titulo_label = QtWidgets.QLabel("Gestión de Escenarios")
        titulo_label.setFont(QtGui.QFont("Helvetica", 16, QtGui.QFont.Bold))
        titulo_label.setAlignment(QtCore.Qt.AlignCenter)
        layout.addWidget(titulo_label)

        # Número de simulaciones
        simulaciones_layout = QtWidgets.QHBoxLayout()
        simulaciones_label = QtWidgets.QLabel("Número de simulaciones:")
        self.num_simulaciones_var_escenarios = QtWidgets.QLineEdit(self.num_simulaciones_var.text())
        self.num_simulaciones_var_escenarios.setFixedWidth(100)
        simulaciones_layout.addWidget(simulaciones_label)
        simulaciones_layout.addWidget(self.num_simulaciones_var_escenarios)
        simulaciones_layout.addStretch()
        layout.addLayout(simulaciones_layout)

        # Lista de Escenarios
        self.scenarios_table = QtWidgets.QTableWidget(0, 2)
        self.scenarios_table.setHorizontalHeaderLabels(["Nombre", "Descripción"])
        self.scenarios_table.horizontalHeader().setStretchLastSection(True)
        self.scenarios_table.setSelectionBehavior(QtWidgets.QAbstractItemView.SelectRows)
        layout.addWidget(self.scenarios_table)

        # Botones de acciones sobre escenarios
        acciones_layout = QtWidgets.QHBoxLayout()
        agregar_scenario_button = QtWidgets.QPushButton("Agregar Escenario")
        editar_scenario_button = QtWidgets.QPushButton("Editar Escenario")
        duplicar_scenario_button = QtWidgets.QPushButton("Duplicar Escenario")
        eliminar_scenario_button = QtWidgets.QPushButton("Eliminar Escenario")

        agregar_scenario_button.clicked.connect(self.agregar_scenario_popup)
        editar_scenario_button.clicked.connect(self.editar_scenario_popup)
        duplicar_scenario_button.clicked.connect(self.duplicar_scenario)
        eliminar_scenario_button.clicked.connect(self.eliminar_scenario)

        acciones_layout.addWidget(agregar_scenario_button)
        acciones_layout.addWidget(editar_scenario_button)
        acciones_layout.addWidget(duplicar_scenario_button)
        acciones_layout.addWidget(eliminar_scenario_button)
        acciones_layout.addStretch()
        layout.addLayout(acciones_layout)

        # Añadir el botón "Ejecutar Simulación"
        simular_button = QtWidgets.QPushButton("Ejecutar Simulación")
        simular_button.setIcon(QtGui.QIcon.fromTheme("system-run"))
        simular_button.clicked.connect(self.ejecutar_simulacion_desde_escenarios)

        layout.addWidget(simular_button)

        # Selección de escenario para simulación
        seleccionar_layout = QtWidgets.QHBoxLayout()
        seleccionar_label = QtWidgets.QLabel("Escenario seleccionado para la simulación:")
        self.selected_scenario_label = QtWidgets.QLabel("Ninguno")
        seleccionar_layout.addWidget(seleccionar_label)
        seleccionar_layout.addWidget(self.selected_scenario_label)
        seleccionar_layout.addStretch()
        layout.addLayout(seleccionar_layout)

        # Conectar doble clic en la tabla para seleccionar escenario
        self.scenarios_table.cellDoubleClicked.connect(self.seleccionar_scenario)

    def ejecutar_simulacion_desde_escenarios(self):
        if self.current_scenario is None:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione un escenario para simular.")
            return
        # Actualizar el número de simulaciones con el valor ingresado en la pestaña "Escenarios"
        self.num_simulaciones_var.setText(self.num_simulaciones_var_escenarios.text())
        # Llamar al método existente de ejecutar_simulacion
        self.ejecutar_simulacion()

    def actualizar_num_simulaciones_escenarios(self, text):
        if self.num_simulaciones_var_escenarios.text() != text:
            self.num_simulaciones_var_escenarios.setText(text)

    def actualizar_num_simulaciones_simulacion(self, text):
        if self.num_simulaciones_var.text() != text:
            self.num_simulaciones_var.setText(text)

    def agregar_scenario_popup(self):
        self.editar_scenario_popup(new=True)

    def editar_scenario_popup(self, new=False, row=None):
        if not new:
            if row is None:
                selected_items = self.scenarios_table.selectedItems()
                if not selected_items:
                    QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione un escenario para editar.")
                    return
                row = selected_items[0].row()
            scenario = self.scenarios[row]
        else:
            scenario = None

        # Crear diálogo
        dialog = QtWidgets.QDialog(self)
        dialog.setWindowTitle("Agregar Escenario" if new else "Editar Escenario")
        dialog_layout = QtWidgets.QVBoxLayout(dialog)

        # Nombre del Escenario
        nombre_layout = QtWidgets.QHBoxLayout()
        nombre_label = QtWidgets.QLabel("Nombre del Escenario:")
        nombre_var = QtWidgets.QLineEdit(scenario.nombre if scenario else "")
        nombre_layout.addWidget(nombre_label)
        nombre_layout.addWidget(nombre_var)
        dialog_layout.addLayout(nombre_layout)

        # Descripción del Escenario
        descripcion_layout = QtWidgets.QHBoxLayout()
        descripcion_label = QtWidgets.QLabel("Descripción:")
        descripcion_var = QtWidgets.QLineEdit(scenario.descripcion if scenario else "")
        descripcion_layout.addWidget(descripcion_label)
        descripcion_layout.addWidget(descripcion_var)
        dialog_layout.addLayout(descripcion_layout)

        # Tabla para mostrar eventos
        eventos_table = QtWidgets.QTableWidget()
        eventos_table.setColumnCount(1)
        eventos_table.setHorizontalHeaderLabels(["Evento"])
        eventos_table.horizontalHeader().setStretchLastSection(True)

        # Obtener la lista de eventos
        self.eventos_scenario = scenario.eventos_riesgo if scenario else [evento.copy() for evento in self.eventos_riesgo]

        # Llenar la tabla con nombres de eventos
        for idx, evento in enumerate(self.eventos_scenario):
            eventos_table.insertRow(idx)
            # Nombre del evento
            nombre_item = QtWidgets.QTableWidgetItem(evento['nombre'])
            nombre_item.setFlags(QtCore.Qt.ItemIsEnabled)
            eventos_table.setItem(idx, 0, nombre_item)

        dialog_layout.addWidget(eventos_table)

        # Función para editar parámetros del evento
        def editar_parametros_evento(row, column):
            evento = self.eventos_scenario[row]
            # Crear diálogo para editar parámetros del evento
            evento_dialog = QtWidgets.QDialog(dialog)
            evento_dialog.setWindowTitle(f"Editar Parámetros del Evento '{evento['nombre']}'")
            evento_layout = QtWidgets.QVBoxLayout(evento_dialog)

            # Sección para la Frecuencia
            freq_group = QtWidgets.QGroupBox("Parámetros de Frecuencia")
            freq_layout = QtWidgets.QFormLayout(freq_group)

            if evento['freq_opcion'] == 1:  # Poisson
                tasa_var = QtWidgets.QLineEdit(str(evento['tasa']))
                freq_layout.addRow("Tasa Media (λ):", tasa_var)
            elif evento['freq_opcion'] == 2:  # Binomial
                n_var = QtWidgets.QLineEdit(str(evento['num_eventos']))
                p_var = QtWidgets.QLineEdit(str(evento['prob_exito']))
                freq_layout.addRow("Número de Eventos (n):", n_var)
                freq_layout.addRow("Probabilidad de Éxito (p):", p_var)
            elif evento['freq_opcion'] == 3:  # Bernoulli
                p_var = QtWidgets.QLineEdit(str(evento['prob_exito']))
                freq_layout.addRow("Probabilidad de Éxito (p):", p_var)

            evento_layout.addWidget(freq_group)

            # Sección para la Severidad
            sev_group = QtWidgets.QGroupBox("Parámetros de Severidad")
            sev_layout = QtWidgets.QFormLayout(sev_group)

            sev_min_var = QtWidgets.QLineEdit(str(evento['sev_minimo']))
            sev_max_var = QtWidgets.QLineEdit(str(evento['sev_maximo']))

            sev_layout.addRow("Valor Mínimo:", sev_min_var)
            if evento['sev_opcion'] != 5:  # Si no es Uniforme
                sev_mas_probable_var = QtWidgets.QLineEdit(str(evento['sev_mas_probable']))
                sev_layout.addRow("Valor Más Probable:", sev_mas_probable_var)
            else:
                sev_mas_probable_var = None
            sev_layout.addRow("Valor Máximo:", sev_max_var)

            evento_layout.addWidget(sev_group)

            # Botones
            evento_buttons = QtWidgets.QDialogButtonBox()
            evento_buttons.setStandardButtons(QtWidgets.QDialogButtonBox.Save | QtWidgets.QDialogButtonBox.Cancel)
            evento_layout.addWidget(evento_buttons)

            evento_buttons.accepted.connect(lambda: guardar_cambios())
            evento_buttons.rejected.connect(evento_dialog.reject)

            def guardar_cambios():
                try:
                    # Actualizar los parámetros de frecuencia
                    if evento['freq_opcion'] == 1:  # Poisson
                        tasa = float(tasa_var.text())
                        if tasa <= 0:
                            raise ValueError("La tasa media (λ) debe ser mayor que cero.")
                        evento['tasa'] = tasa
                    elif evento['freq_opcion'] == 2:  # Binomial
                        n = int(n_var.text())
                        p = float(p_var.text())
                        if n <= 0:
                            raise ValueError("El número de eventos (n) debe ser mayor que cero.")
                        if not 0 <= p <= 1:
                            raise ValueError("La probabilidad de éxito (p) debe estar entre 0 y 1.")
                        evento['num_eventos'] = n
                        evento['prob_exito'] = p
                    elif evento['freq_opcion'] == 3:  # Bernoulli
                        p = float(p_var.text())
                        if not 0 <= p <= 1:
                            raise ValueError("La probabilidad de éxito (p) debe estar entre 0 y 1.")
                        evento['prob_exito'] = p

                    # Actualizar la distribución de frecuencia
                    evento['dist_frecuencia'] = generar_distribucion_frecuencia(
                        evento['freq_opcion'],
                        tasa=evento.get('tasa'),
                        num_eventos_posibles=evento.get('num_eventos'),
                        probabilidad_exito=evento.get('prob_exito')
                    )

                    # Actualizar los parámetros de severidad
                    sev_minimo = float(sev_min_var.text())
                    sev_maximo = float(sev_max_var.text())
                    if evento['sev_opcion'] != 5:  # Si no es Uniforme
                        sev_mas_probable = float(sev_mas_probable_var.text())
                    else:
                        sev_mas_probable = None

                    evento['sev_minimo'] = sev_minimo
                    evento['sev_mas_probable'] = sev_mas_probable
                    evento['sev_maximo'] = sev_maximo

                    # Actualizar la distribución de severidad
                    evento['dist_severidad'] = generar_distribucion_severidad(
                        evento['sev_opcion'],
                        evento['sev_minimo'],
                        evento['sev_mas_probable'],
                        evento['sev_maximo']
                    )

                    evento_dialog.accept()
                except ValueError as ve:
                    QtWidgets.QMessageBox.critical(evento_dialog, "Error", str(ve))
                except Exception as e:
                    QtWidgets.QMessageBox.critical(evento_dialog, "Error", f"No se pudo guardar los cambios: {e}")

            evento_dialog.exec_()

        # Conectar doble clic en la tabla para editar parámetros específicos
        eventos_table.cellDoubleClicked.connect(editar_parametros_evento)

        # Botones del diálogo principal para guardar o cancelar el escenario
        buttons = QtWidgets.QDialogButtonBox()
        buttons.setStandardButtons(QtWidgets.QDialogButtonBox.Save | QtWidgets.QDialogButtonBox.Cancel)
        dialog_layout.addWidget(buttons)

        buttons.accepted.connect(lambda: self.guardar_scenario(dialog, new, row, nombre_var, descripcion_var))
        buttons.rejected.connect(dialog.reject)

        # Mostrar el diálogo
        dialog.exec_()

    def guardar_scenario(self, dialog, new, row, nombre_var, descripcion_var):
        try:
            nombre_scenario = nombre_var.text().strip()
            descripcion_scenario = descripcion_var.text().strip()

            if not nombre_scenario:
                raise ValueError("El nombre del escenario no puede estar vacío.")

            # Crear instancia del escenario
            scenario = Scenario(nombre_scenario, descripcion_scenario)
            scenario.eventos_riesgo = self.eventos_scenario

            # Guardar o actualizar el escenario
            if new:
                self.scenarios.append(scenario)
                row_position = self.scenarios_table.rowCount()
                self.scenarios_table.insertRow(row_position)
                self.scenarios_table.setItem(row_position, 0, QtWidgets.QTableWidgetItem(nombre_scenario))
                self.scenarios_table.setItem(row_position, 1, QtWidgets.QTableWidgetItem(descripcion_scenario))
            else:
                self.scenarios[row] = scenario
                self.scenarios_table.setItem(row, 0, QtWidgets.QTableWidgetItem(nombre_scenario))
                self.scenarios_table.setItem(row, 1, QtWidgets.QTableWidgetItem(descripcion_scenario))

            dialog.accept()

        except ValueError as ve:
            QtWidgets.QMessageBox.critical(self, "Error", str(ve))
        except Exception as e:
            QtWidgets.QMessageBox.critical(self, "Error", f"No se pudo guardar el escenario: {e}")

    def eliminar_scenario(self):
        selected_items = self.scenarios_table.selectionModel().selectedRows()
        if not selected_items:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione al menos un escenario para eliminar.")
            return

        # Confirmar eliminación múltiple
        respuesta = QtWidgets.QMessageBox.question(
            self,
            "Eliminar Escenario(s)",
            f"¿Estás seguro de que deseas eliminar {len(selected_items)} escenario(s)?",
            QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No
        )

        if respuesta == QtWidgets.QMessageBox.Yes:
            # Ordenar los índices de fila en orden descendente para evitar problemas al eliminar filas
            rows = sorted([index.row() for index in selected_items], reverse=True)
            for row in rows:
                scenario_to_delete = self.scenarios[row]
                # Si el escenario a eliminar es el escenario seleccionado, deseleccionarlo
                if self.current_scenario == scenario_to_delete:
                    self.current_scenario = None
                    self.selected_scenario_label.setText("Ninguno")
                del self.scenarios[row]
                self.scenarios_table.removeRow(row)
            QtWidgets.QMessageBox.information(self, "Eliminar Escenario(s)", "El(los) escenario(s) seleccionado(s) han sido eliminados.")

    def duplicar_scenario(self):
        selected_items = self.scenarios_table.selectionModel().selectedRows()
        if not selected_items:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione un escenario para duplicar.")
            return
        if len(selected_items) > 1:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "Seleccione solo un escenario para duplicar.")
            return
        row = selected_items[0].row()
        scenario_original = self.scenarios[row]

        # Crear una copia profunda del escenario
        import copy
        escenario_nuevo = copy.deepcopy(scenario_original)

        # Modificar el nombre y descripción del nuevo escenario
        escenario_nuevo.nombre = escenario_nuevo.nombre + " (Copia)"
        if escenario_nuevo.descripcion:
            escenario_nuevo.descripcion = escenario_nuevo.descripcion + " (Duplicado)"
        else:
            escenario_nuevo.descripcion = "Duplicado del escenario original"

        # Generar nuevos IDs para los eventos dentro del escenario duplicado
        id_original_a_nuevo = {}
        for evento in escenario_nuevo.eventos_riesgo:
            nuevo_id = str(uuid.uuid4())
            id_original_a_nuevo[evento['id']] = nuevo_id
            evento['id'] = nuevo_id

        # Actualizar las dependencias de los eventos duplicados
        for evento in escenario_nuevo.eventos_riesgo:
            # Manejar la nueva estructura de vínculos
            if 'vinculos' in evento:
                vinculos_actualizados = []
                for vinculo in evento.get('vinculos', []):
                    padre_id = vinculo['id_padre']
                    tipo = vinculo['tipo']
                    # Si el evento padre fue duplicado, usamos el nuevo ID
                    if padre_id in id_original_a_nuevo:
                        vinculos_actualizados.append({
                            'id_padre': id_original_a_nuevo[padre_id],
                            'tipo': tipo
                        })
                    else:
                        # Si no, mantenemos el ID original
                        vinculos_actualizados.append({
                            'id_padre': padre_id,
                            'tipo': tipo
                        })
                evento['vinculos'] = vinculos_actualizados

            # Compatibilidad con formato antiguo
            elif 'eventos_padres' in evento:
                eventos_padres_originales = evento.get('eventos_padres', [])
                eventos_padres_actualizados = []
                for padre_id in eventos_padres_originales:
                    # Si el evento padre fue duplicado, usamos el nuevo ID
                    if padre_id in id_original_a_nuevo:
                        eventos_padres_actualizados.append(id_original_a_nuevo[padre_id])
                    else:
                        # Si no, mantenemos el ID original
                        eventos_padres_actualizados.append(padre_id)
                evento['eventos_padres'] = eventos_padres_actualizados

        # Verificar si el escenario duplicado introduce ciclos
        if self.tiene_ciclo(escenario_nuevo.eventos_riesgo):
            QtWidgets.QMessageBox.critical(self, "Error", "La duplicación de este escenario genera una dependencia cíclica.")
            return

        # Agregar el escenario duplicado a la lista y a la tabla
        self.scenarios.append(escenario_nuevo)
        row_position = self.scenarios_table.rowCount()
        self.scenarios_table.insertRow(row_position)
        self.scenarios_table.setItem(row_position, 0, QtWidgets.QTableWidgetItem(escenario_nuevo.nombre))
        self.scenarios_table.setItem(row_position, 1, QtWidgets.QTableWidgetItem(escenario_nuevo.descripcion))

        QtWidgets.QMessageBox.information(self, "Duplicar Escenario", "El escenario ha sido duplicado exitosamente.")

    def seleccionar_scenario(self, row, column):
        scenario = self.scenarios[row]
        self.current_scenario = scenario
        self.selected_scenario_label.setText(scenario.nombre)
        QtWidgets.QMessageBox.information(self, "Escenario Seleccionado", f"Has seleccionado el escenario: {scenario.nombre}")

    def generar_figuras(self, perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento, eventos_riesgo):
        """Genera las figuras de los gráficos para el reporte PDF."""
        figuras = []

        sns.set(style="whitegrid")

        # Gráfico 1: Distribución de pérdidas agregadas
        fig1 = Figure()
        ax1 = fig1.add_subplot(111)
        sns.histplot(perdidas_totales, bins=50, kde=True, color='#1f77b4', edgecolor='black', ax=ax1)
        media = np.mean(perdidas_totales)
        var_90 = np.percentile(perdidas_totales, 90)
        ax1.axvline(x=media, color='red', linestyle='-', linewidth=2, label=f'Media: {currency_format(media)}')
        ax1.axvline(x=var_90, color='green', linestyle='--', linewidth=2, label=f'P90: {currency_format(var_90)}')
        ax1.set_title('Distribución de Pérdidas Agregadas')
        ax1.set_xlabel('Pérdida Total')
        ax1.set_ylabel('Frecuencia')
        ax1.legend(fontsize=8)
        ax1.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
        figuras.append(fig1)

        # Gráfico 2: Distribución de pérdidas agregadas (sin ceros)
        perdidas_totales_sin_cero = perdidas_totales[perdidas_totales != 0]
        if len(perdidas_totales_sin_cero) > 0:
            fig2 = Figure()
            ax2 = fig2.add_subplot(111)
            sns.histplot(perdidas_totales_sin_cero, bins=50, kde=True, color='#1f77b4', edgecolor='black', ax=ax2)
            media_sin_cero = np.mean(perdidas_totales_sin_cero)
            var_90_sin_cero = np.percentile(perdidas_totales_sin_cero, 90)
            ax2.axvline(x=media_sin_cero, color='red', linestyle='-', linewidth=2, label=f'Media: {currency_format(media_sin_cero)}')
            ax2.axvline(x=var_90_sin_cero, color='green', linestyle='--', linewidth=2, label=f'P90: {currency_format(var_90_sin_cero)}')
            ax2.set_title('Distribución de Pérdidas Agregadas (Sin eventos en cero)')
            ax2.set_xlabel('Pérdida Total')
            ax2.set_ylabel('Frecuencia')
            ax2.legend(fontsize=8)
            ax2.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
            figuras.append(fig2)

        # Gráfico 3: Curva de Excedencia
        fig3 = Figure()
        ax3 = fig3.add_subplot(111)
        sorted_losses = np.sort(perdidas_totales)
        exceedance_prob = 1.0 - np.arange(1, len(sorted_losses) + 1) / len(sorted_losses)
        ax3.plot(sorted_losses, exceedance_prob, color='#ff7f0e', linewidth=2)
        ax3.set_title('Curva de Excedencia de Pérdidas Agregadas')
        ax3.set_xlabel('Pérdida Total')
        ax3.set_ylabel('Probabilidad de Excedencia')
        ax3.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax3.yaxis.set_major_formatter(FuncFormatter(percentage_formatter))
        figuras.append(fig3)


        # Gráfico 4: Histograma de Frecuencia de Eventos
        if frecuencias_totales.max() > 0:
            fig4 = Figure()
            canvas4 = FigureCanvas(fig4)
            ax4 = fig4.add_subplot(111)
            bins = range(0, int(frecuencias_totales.max()) + 2)
            sns.histplot(frecuencias_totales, bins=bins, color='#1f77b4', edgecolor='black', ax=ax4)
            ax4.set_title('Histograma de Frecuencia de Eventos')
            ax4.set_xlabel('Número de Eventos')
            ax4.set_ylabel('Frecuencia')
            figuras.append(fig4)

        # Gráfico 5: Dispersión Frecuencia vs. Pérdidas
        if np.std(frecuencias_totales) > 0 and np.std(perdidas_totales) > 0:
            fig5 = Figure()
            canvas5 = FigureCanvas(fig5)
            ax5 = fig5.add_subplot(111)
            ax5.scatter(frecuencias_totales, perdidas_totales, alpha=0.5)
            ax5.set_title('Dispersión de Frecuencia vs. Pérdida Total')
            ax5.set_xlabel('Frecuencia Total de Eventos')
            ax5.set_ylabel('Pérdida Total')
            ax5.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
            figuras.append(fig5)

        # Gráfico 6: Comparación de Pérdidas por Evento de Riesgo
        fig6 = Figure()
        canvas6 = FigureCanvas(fig6)
        ax6 = fig6.add_subplot(111)
        datos_plot = False
        for idx, perdidas_evento in enumerate(perdidas_por_evento):
            nombre_evento = eventos_riesgo[idx]['nombre']
            if np.std(perdidas_evento) > 0:
                sns.kdeplot(perdidas_evento, label=nombre_evento, ax=ax6, bw_method='silverman')
                datos_plot = True
        if datos_plot:
            ax6.set_title('Comparación entre Eventos de Riesgo')
            ax6.set_xlabel('Pérdida')
            ax6.set_ylabel('Densidad')
            ax6.legend(fontsize=8)
            ax6.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
        figuras.append(fig6)

        # Gráfico 7: Gráfico de Tornado - Contribución de Eventos de Riesgo
        contribuciones = []
        nombres_eventos = []
        for idx, perdidas_evento in enumerate(perdidas_por_evento):
            contribucion = np.mean(perdidas_evento)
            contribuciones.append(contribucion)
            nombre_evento = eventos_riesgo[idx]['nombre']
            nombres_eventos.append(nombre_evento)

        if any(c > 0 for c in contribuciones):
            fig7 = Figure()
            canvas7 = FigureCanvas(fig7)
            ax7 = fig7.add_subplot(111)
            tornado_df = pd.DataFrame({
                'Evento de Riesgo': nombres_eventos,
                'Contribución Promedio': contribuciones
            })
            tornado_df = tornado_df[tornado_df['Contribución Promedio'] > 0]
            tornado_df.sort_values('Contribución Promedio', inplace=True, ascending=True)
            ax7.barh(tornado_df['Evento de Riesgo'], tornado_df['Contribución Promedio'], color='#1f77b4', edgecolor='black')
            ax7.set_title('Gráfico de Tornado - Contribución de Eventos de Riesgo')
            ax7.set_xlabel('Contribución Promedio a la Pérdida Media Total')
            ax7.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
            figuras.append(fig7)

        # Gráfico 8: Box Plots por Evento de Riesgo
        fig8 = Figure()
        ax8 = fig8.add_subplot(111)
        datos_perdidas = [perdidas_evento for perdidas_evento in perdidas_por_evento]
        nombres_eventos = [evento['nombre'] for evento in eventos_riesgo]

        # Crear el box plot
        ax8.boxplot(datos_perdidas, labels=nombres_eventos, vert=True, patch_artist=True)
        ax8.set_title('Distribución de Pérdidas por Evento de Riesgo')
        ax8.set_xlabel('Evento de Riesgo')
        ax8.set_ylabel('Pérdida')
        ax8.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax8.set_xticklabels(nombres_eventos, rotation=45, ha='right')
        fig8.tight_layout()
        figuras.append(fig8)

        # Cálculo de perdidas_cola
        percentil_80 = np.percentile(perdidas_totales, 80)
        perdidas_cola = perdidas_totales[perdidas_totales >= percentil_80]

        # Gráfico 9: Cola de Pérdidas (Tail Risk)
        fig10 = Figure()
        ax10 = fig10.add_subplot(111)
        sns.histplot(perdidas_cola, bins=30, kde=True, color='#1f77b4', edgecolor='black', ax=ax10)
        ax10.set_title('Cola de Pérdidas (Percentil 80 al 100)')
        ax10.set_xlabel('Pérdida Total')
        ax10.set_ylabel('Frecuencia')
        ax10.xaxis.set_major_formatter(FuncFormatter(currency_formatter))

        # Añadir líneas verticales para percentiles clave
        percentiles_cola = np.percentile(perdidas_totales, [90, 95, 99])
        colores_percentiles = ['green', 'orange', 'red']
        labels_percentiles = ['P90', 'P95', 'P99']
        for p, color, label in zip(percentiles_cola, colores_percentiles, labels_percentiles):
            ax10.axvline(x=p, color=color, linestyle='--', linewidth=2, label=f'{label}: {currency_format(p)}')

        ax10.legend()
        fig10.tight_layout()
        figuras.append(fig10)

        return figuras

    def graficar_resultados(self, perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento, eventos_riesgo):
        """Genera y muestra los gráficos en la pestaña de Resultados."""
        # Limpiar las pestañas de gráficos anteriores
        self.graficos_tab_widget.clear()

        sns.set(style="whitegrid")

        # Ajuste de bins usando la regla de Freedman-Diaconis
        bin_width = 2 * stats.iqr(perdidas_totales) / (len(perdidas_totales) ** (1/3))
        bins = int((perdidas_totales.max() - perdidas_totales.min()) / bin_width)

        # Gráfico 1: Distribución de pérdidas agregadas
        fig1 = Figure()
        canvas1 = FigureCanvas(fig1)
        ax1 = fig1.add_subplot(111)
        sns.histplot(perdidas_totales, bins=bins, kde=True, color='#4c72b0', edgecolor='black', ax=ax1)
        media = np.mean(perdidas_totales)
        mediana = np.median(perdidas_totales)
        var_90 = np.percentile(perdidas_totales, 90)
        var_99 = np.percentile(perdidas_totales, 99)
        ax1.axvline(x=media, color='red', linestyle='-', linewidth=2, label=f'Media: {currency_format(media)}')
        ax1.axvline(x=mediana, color='purple', linestyle='-.', linewidth=2, label=f'Mediana: {currency_format(mediana)}')
        ax1.axvline(x=var_90, color='green', linestyle='--', linewidth=2, label=f'P90: {currency_format(var_90)}')
        ax1.axvline(x=var_99, color='orange', linestyle=':', linewidth=2, label=f'P99: {currency_format(var_99)}')
        ax1.set_title('Rango de Impacto Económico Anual Esperado')
        ax1.set_xlabel('Pérdida Total')
        ax1.set_ylabel('Frecuencia')
        ax1.legend(fontsize=8)
        ax1.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'.replace(",", ".")))
        tab1 = QtWidgets.QWidget()
        layout1 = QtWidgets.QVBoxLayout(tab1)
        layout1.addWidget(canvas1)
        self.graficos_tab_widget.addTab(tab1, "Distribución Agregada")

        # Gráfico 2: Distribución de pérdidas agregadas (sin ceros)
        perdidas_totales_sin_cero = perdidas_totales[perdidas_totales != 0]
        # Cálculo del porcentaje de casos no cero
        porcentaje_no_cero = (len(perdidas_totales_sin_cero) / len(perdidas_totales)) * 100
        if len(perdidas_totales_sin_cero) > 0:
            fig2 = Figure()
            canvas2 = FigureCanvas(fig2)
            ax2 = fig2.add_subplot(111)
            sns.histplot(perdidas_totales_sin_cero, bins=50, kde=True, color='#1f77b4', edgecolor='black', ax=ax2)
            media_sin_cero = np.mean(perdidas_totales_sin_cero)
            var_90_sin_cero = np.percentile(perdidas_totales_sin_cero, 90)
            ax2.axvline(x=media_sin_cero, color='red', linestyle='-', linewidth=2, label=f'Media: {currency_format(media_sin_cero)}')
            ax2.axvline(x=var_90_sin_cero, color='green', linestyle='--', linewidth=2, label=f'P90: {currency_format(var_90_sin_cero)}')
            ax2.set_title(f'Rango de Impacto Económico Anual Esperado (Sin eventos en cero)\n{porcentaje_no_cero:.2f}% de las simulaciones')
            ax2.set_xlabel('Pérdida Total')
            ax2.set_ylabel('Frecuencia')
            ax2.legend(fontsize=8)
            ax2.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
            tab2 = QtWidgets.QWidget()
            layout2 = QtWidgets.QVBoxLayout(tab2)
            layout2.addWidget(canvas2)
            self.graficos_tab_widget.addTab(tab2, "Distribución sin eventos en cero")

        # Gráfico 3: Curva de Excedencia
        fig3 = Figure()
        canvas3 = FigureCanvas(fig3)
        ax3 = fig3.add_subplot(111)
        percentiles = np.arange(100)

        # Calcular las pérdidas correspondientes a cada percentil
        loss_values = np.percentile(perdidas_totales, percentiles)

        # Trazar los percentiles en el eje X contra las pérdidas en el eje Y
        ax3.plot(percentiles, loss_values, color='#ff7f0e', linewidth=2)

        # Cálculo de P50, P90 y P99
        p50 = np.percentile(perdidas_totales, 50)
        p90 = np.percentile(perdidas_totales, 90)
        p99 = np.percentile(perdidas_totales, 99)

        # Líneas horizontales para los percentiles
        ax3.axhline(y=p50, color='green', linestyle='--', linewidth=1.5)  # P50 en verde
        ax3.axhline(y=p90, color='red', linestyle='--', linewidth=1.5)    # P90 en rojo
        ax3.axhline(y=p99, color='black', linestyle='--', linewidth=1.5)  # P99 en negro

        # Puntos para marcar los percentiles en la curva
        ax3.scatter(50, p50, color='green', s=50, zorder=5)  # P50
        ax3.scatter(90, p90, color='red', s=50, zorder=5)    # P90
        ax3.scatter(99, p99, color='black', s=50, zorder=5)  # P99

        # Etiquetas de texto sin flechas, posicionadas estratégicamente
        ax3.text(40, p50 * 1.1, f'P50: {currency_format(p50)}', color='green', fontsize=9)
        ax3.text(80, p90 * 1.1, f'P90: {currency_format(p90)}', color='red', fontsize=9)
        ax3.text(89, p99 * 1.1, f'P99: {currency_format(p99)}', color='black', fontsize=9)

        ax3.set_title('Curva de Excedencia de Pérdidas Agregadas')
        ax3.set_xlabel('Percentiles')
        ax3.set_ylabel('Pérdida Total')
        ax3.set_xticks(np.append(np.arange(0, 100, 10), 99))
        ax3.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax3.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x)}'))
        fig3.tight_layout()

        tab3 = QtWidgets.QWidget()
        layout3 = QtWidgets.QVBoxLayout(tab3)
        layout3.addWidget(canvas3)
        self.graficos_tab_widget.addTab(tab3, "Curva de Excedencia")

        # Gráfico: Box Plot de Pérdida Agregada
        fig_boxplot = Figure()
        canvas_boxplot = FigureCanvas(fig_boxplot)
        ax_boxplot = fig_boxplot.add_subplot(111)

        # Crear el box plot con diseño mejorado
        box = ax_boxplot.boxplot([perdidas_totales], labels=['Pérdida Agregada'],
                                patch_artist=True, showfliers=True,
                                whiskerprops={'color':'black', 'linestyle':'-'},
                                medianprops={'color':'red', 'linewidth':2})

        # Personalizar colores
        for patch in box['boxes']:
            patch.set_facecolor('#4c72b0')
            patch.set_alpha(0.7)

        # Añadir etiquetas con estadísticas clave
        mean_val = np.mean(perdidas_totales)
        median_val = np.median(perdidas_totales)
        p25 = np.percentile(perdidas_totales, 25)
        p75 = np.percentile(perdidas_totales, 75)

        stats_text = f"Media: {currency_format(mean_val)}\n"
        stats_text += f"Mediana: {currency_format(median_val)}\n"
        stats_text += f"Q1: {currency_format(p25)}\n"
        stats_text += f"Q3: {currency_format(p75)}"

        ax_boxplot.text(1.15, median_val, stats_text, va='center', ha='left')

        ax_boxplot.set_title('Distribución de Pérdida Agregada')
        ax_boxplot.set_ylabel('Pérdida')
        ax_boxplot.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        fig_boxplot.tight_layout()

        tab4 = QtWidgets.QWidget()
        layout_boxplot = QtWidgets.QVBoxLayout(tab4)
        layout_boxplot.addWidget(canvas_boxplot)
        self.graficos_tab_widget.addTab(tab4, "Box Plot Agregado")

        # Gráfico: Violin Plot para Pérdida Agregada
        fig_violin = Figure()
        canvas_violin = FigureCanvas(fig_violin)
        ax_violin = fig_violin.add_subplot(111)

        # Preparar datos en formato adecuado para violinplot
        violin_data = [perdidas_totales]

        # Crear violin plot
        parts = ax_violin.violinplot(violin_data, showmedians=True, showextrema=True)

        # Personalizar el violin plot
        for pc in parts['bodies']:
            pc.set_facecolor('#4c72b0')
            pc.set_alpha(0.7)
            pc.set_edgecolor('black')

        # Añadir etiquetas con percentiles clave
        percentiles = [10, 25, 50, 75, 90, 95, 99]
        percentiles_val = np.percentile(perdidas_totales, percentiles)

        for i, p in enumerate(percentiles):
            ax_violin.axhline(y=percentiles_val[i], color='green', linestyle='--', alpha=0.5, linewidth=1)
            ax_violin.text(1.1, percentiles_val[i], f"P{p}: {currency_format(percentiles_val[i])}",
                          va='center', ha='left', fontsize=8)

        ax_violin.set_title('Distribución Detallada de Pérdida Agregada')
        ax_violin.set_ylabel('Pérdida')
        ax_violin.set_xticks([1])
        ax_violin.set_xticklabels(['Pérdida Agregada'])
        ax_violin.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        fig_violin.tight_layout()

        tab5 = QtWidgets.QWidget()
        layout_violin = QtWidgets.QVBoxLayout(tab5)
        layout_violin.addWidget(canvas_violin)
        self.graficos_tab_widget.addTab(tab5, "Violin Plot")

        # Gráfico: Split Violin para comparar distribución completa vs cola
        fig_split = Figure()
        canvas_split = FigureCanvas(fig_split)
        ax_split = fig_split.add_subplot(111)

        # Obtener la cola de distribución (últimos 20%)
        percentil_80 = np.percentile(perdidas_totales, 80)
        perdidas_cola = perdidas_totales[perdidas_totales >= percentil_80]

        # Preparar datos para el split violin
        split_data = [perdidas_totales, perdidas_cola]
        positions = [1, 2]
        labels = ['Distribución Completa', 'Cola (P80-P100)']

        # Crear violin split
        parts = ax_split.violinplot(split_data, positions, showmedians=True, showextrema=True)

        # Personalizar colores
        for i, pc in enumerate(parts['bodies']):
            if i == 0:
                pc.set_facecolor('#4c72b0')
            else:
                pc.set_facecolor('#de425b')
            pc.set_alpha(0.7)
            pc.set_edgecolor('black')

        # Añadir textos con estadísticas
        for i, data in enumerate(split_data):
            mean_val = np.mean(data)
            median_val = np.median(data)
            stats_text = f"Media: {currency_format(mean_val)}\n"
            stats_text += f"Mediana: {currency_format(median_val)}"
            ax_split.text(positions[i] + 0.3, median_val, stats_text, va='center', ha='left', fontsize=8)

        ax_split.set_title('Comparativa de Distribución Completa vs Cola')
        ax_split.set_ylabel('Pérdida')
        ax_split.set_xticks(positions)
        ax_split.set_xticklabels(labels)
        ax_split.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        fig_split.tight_layout()

        tab6 = QtWidgets.QWidget()
        layout_split = QtWidgets.QVBoxLayout(tab6)
        layout_split.addWidget(canvas_split)
        self.graficos_tab_widget.addTab(tab6, "Comparativa")

        # Gráfico 4: Histograma de Frecuencia de Eventos
        if frecuencias_totales.max() > 0:
            fig4 = Figure()
            canvas4 = FigureCanvas(fig4)
            ax4 = fig4.add_subplot(111)

            # Contar la frecuencia de cada número de eventos
            frecuencia_counts = np.bincount(frecuencias_totales.astype(int))
            x_values = np.arange(len(frecuencia_counts))

            # Crear el gráfico de barras
            sns.barplot(x=x_values, y=frecuencia_counts, color='#1f77b4', edgecolor='black', ax=ax4)

            # Ajustar las etiquetas del eje X según la cantidad de valores
            max_freq = len(frecuencia_counts)

            if max_freq <= 15:
                # Si hay pocos valores, mostrarlos todos
                ax4.set_xticks(range(max_freq))
                ax4.set_xticklabels(range(max_freq))
            elif max_freq <= 30:
                # Si hay una cantidad moderada, mostrar cada segundo valor
                ax4.set_xticks(range(0, max_freq, 2))
                ax4.set_xticklabels(range(0, max_freq, 2))
            elif max_freq <= 100:
                # Si hay muchos valores, mostrar cada quinto valor
                step = 5
                ax4.set_xticks(range(0, max_freq, step))
                ax4.set_xticklabels(range(0, max_freq, step))
            else:
                # Para cantidades muy grandes, usar una escala logarítmica o mostrar deciles
                step = max(1, max_freq // 10)
                ticks = list(range(0, max_freq, step))
                if max_freq - 1 not in ticks:
                    ticks.append(max_freq - 1)  # Asegurarse de que se muestre el último valor
                ax4.set_xticks(ticks)
                ax4.set_xticklabels([str(x) for x in ticks])

            # Mejorar formato y visibilidad
            ax4.set_title('Distribución de Frecuencia de Eventos')
            ax4.set_xlabel('Número de Eventos')
            ax4.set_ylabel('Frecuencia')

            # Asegurarse de que haya suficiente espacio para las etiquetas
            fig4.tight_layout()

            tab7 = QtWidgets.QWidget()
            layout4 = QtWidgets.QVBoxLayout(tab7)
            layout4.addWidget(canvas4)
            self.graficos_tab_widget.addTab(tab7, "Frecuencia de Eventos")

        # Gráfico 5: Dispersión Frecuencia vs. Pérdidas
#        if np.std(frecuencias_totales) > 0 and np.std(perdidas_totales) > 0:
#            fig5 = Figure()
#            canvas5 = FigureCanvas(fig5)
#            ax5 = fig5.add_subplot(111)
#            ax5.scatter(frecuencias_totales, perdidas_totales, alpha=0.5)
#            ax5.set_title('Dispersión de Frecuencia vs. Pérdida Total')
#            ax5.set_xlabel('Frecuencia Total de Eventos')
#            ax5.set_ylabel('Pérdida Total')
#            ax5.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
#            tab5 = QtWidgets.QWidget()
#            layout5 = QtWidgets.QVBoxLayout(tab5)
#            layout5.addWidget(canvas5)
#            self.graficos_tab_widget.addTab(tab5, "Dispersión de Resultados")

        # Gráfico 5: Dispersión Frecuencia vs. Pérdidas
        if np.std(frecuencias_totales) > 0 and np.std(perdidas_totales) > 0:
            fig5 = Figure()
            canvas5 = FigureCanvas(fig5)
            ax5 = fig5.add_subplot(111)
            sns.scatterplot(x=frecuencias_totales + np.random.uniform(-0.2, 0.2, size=frecuencias_totales.size),
                            y=perdidas_totales,
                            alpha=0.5, s=20, ax=ax5, hue=frecuencias_totales, palette='viridis', legend=False)
            sns.regplot(x=frecuencias_totales, y=perdidas_totales, scatter=False, ax=ax5, color='red', line_kws={'linewidth':1})
            ax5.set_title('Dispersión de Frecuencia vs. Pérdida Total')
            ax5.set_xlabel('Frecuencia Total de Eventos')
            ax5.set_ylabel('Pérdida Total')
            ax5.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
            tab8 = QtWidgets.QWidget()
            layout5 = QtWidgets.QVBoxLayout(tab8)
            layout5.addWidget(canvas5)
            self.graficos_tab_widget.addTab(tab8, "Dispersión")


        # Gráfico 6: Comparación de Pérdidas por Evento de Riesgo
        fig6 = Figure()
        canvas6 = FigureCanvas(fig6)
        ax6 = fig6.add_subplot(111)
        datos_plot = False
        for idx, perdidas_evento in enumerate(perdidas_por_evento):
            nombre_evento = eventos_riesgo[idx]['nombre']
            if np.std(perdidas_evento) > 0:
                sns.kdeplot(perdidas_evento, label=nombre_evento, ax=ax6, bw_method='silverman')
                datos_plot = True
        if datos_plot:
            ax6.set_title('Comparación entre Eventos de Riesgo')
            ax6.set_xlabel('Pérdida')
            ax6.set_ylabel('Densidad')
            ax6.legend(fontsize=8)
            ax6.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
            tab9 = QtWidgets.QWidget()
            layout6 = QtWidgets.QVBoxLayout(tab9)
            layout6.addWidget(canvas6)
            self.graficos_tab_widget.addTab(tab9, "Distribución por Evento")

        # Gráfico 7: Gráfico de Tornado - Contribución de Eventos de Riesgo
        contribuciones = []
        nombres_eventos = []
        for idx, perdidas_evento in enumerate(perdidas_por_evento):
            contribucion = np.mean(perdidas_evento)
            contribuciones.append(contribucion)
            nombre_evento = eventos_riesgo[idx]['nombre']
            nombres_eventos.append(nombre_evento)

        if any(c > 0 for c in contribuciones):
            fig7 = Figure()
            canvas7 = FigureCanvas(fig7)
            ax7 = fig7.add_subplot(111)
            tornado_df = pd.DataFrame({
                'Evento de Riesgo': nombres_eventos,
                'Contribución Promedio': contribuciones
            })
            tornado_df = tornado_df[tornado_df['Contribución Promedio'] > 0]
            tornado_df['Porcentaje'] = (tornado_df['Contribución Promedio'] / tornado_df['Contribución Promedio'].sum()) * 100
            tornado_df.sort_values('Contribución Promedio', inplace=True, ascending=True)
            bars = ax7.barh(tornado_df['Evento de Riesgo'], tornado_df['Contribución Promedio'], color='#4c72b0', edgecolor='black')
            ax7.set_title('Principales Fuentes de Pérdida Esperada (Promedio)')
            ax7.set_xlabel('Contribución a la Pérdida Media Total')
            ax7.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
            # Añadir etiquetas con porcentajes
            for bar, porcentaje in zip(bars, tornado_df['Porcentaje']):
                width = bar.get_width()
                ax7.text(width + max(contribuciones) * 0.01, bar.get_y() + bar.get_height() / 2,
                        '{:.1f}%'.format(porcentaje), va='center')
            tab10 = QtWidgets.QWidget()
            layout7 = QtWidgets.QVBoxLayout(tab10)
            layout7.addWidget(canvas7)
            self.graficos_tab_widget.addTab(tab10, "Gráfico de Tornado")

        # Gráfico 8: Box Plots por Evento de Riesgo
        fig8 = Figure()
        canvas8 = FigureCanvas(fig8)
        ax8 = fig8.add_subplot(111)
        datos_perdidas = [perdidas_evento for perdidas_evento in perdidas_por_evento]
        nombres_eventos = [evento['nombre'] for evento in eventos_riesgo]

        # Crear el box plot sin outliers
        box = ax8.boxplot(datos_perdidas, labels=nombres_eventos, vert=True, patch_artist=True, showfliers=False)
        colors = sns.color_palette("pastel", len(datos_perdidas))
        for patch, color in zip(box['boxes'], colors):
            patch.set_facecolor(color)

        ax8.set_title('Distribución de Pérdidas por Evento de Riesgo')
        ax8.set_xlabel('Evento de Riesgo')
        ax8.set_ylabel('Pérdida')
        ax8.yaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax8.set_xticklabels(nombres_eventos, rotation=45, ha='right')
        fig8.tight_layout()

        tab11 = QtWidgets.QWidget()
        layout8 = QtWidgets.QVBoxLayout(tab11)
        layout8.addWidget(canvas8)
        self.graficos_tab_widget.addTab(tab11, "Pérdidas por Evento")

        # Cálculo de perdidas_cola
        percentil_80 = np.percentile(perdidas_totales, 80)
        perdidas_cola = perdidas_totales[perdidas_totales >= percentil_80]

        # Gráfico 9: Cola de Pérdidas (Tail Risk)
        fig10 = Figure()
        canvas10 = FigureCanvas(fig10)
        ax10 = fig10.add_subplot(111)
        sns.histplot(perdidas_cola, bins=30, kde=True, color='#de425b', edgecolor='black', ax=ax10)
        ax10.set_title('Cola de Pérdidas (Percentil 80 al 100)')
        ax10.set_xlabel('Pérdida Total')
        ax10.set_ylabel('Frecuencia')
        ax10.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax10.yaxis.set_major_formatter(FuncFormatter(lambda y, _: '{:,}'.format(int(y)).replace(",", ".")))
        percentiles_cola = np.percentile(perdidas_totales, [90, 95, 99])
        colores_percentiles = ['green', 'orange', 'red']
        labels_percentiles = ['P90', 'P95', 'P99']
        for p, color, label in zip(percentiles_cola, colores_percentiles, labels_percentiles):
            ax10.axvline(x=p, color=color, linestyle='--', linewidth=2, label=f'{label}: {currency_format(p)}')
            ax10.text(p, ax10.get_ylim()[1]*0.9, f'{label}', color=color, rotation=90, va='top', ha='right')
        ax10.legend()
        fig10.tight_layout()
        tab12 = QtWidgets.QWidget()
        layout10 = QtWidgets.QVBoxLayout(tab12)
        layout10.addWidget(canvas10)
        self.graficos_tab_widget.addTab(tab12, "Cola de Pérdidas")

# --- INICIO: Nuevo Gráfico - Mapa de Riesgos (Scatterplot - V3: Color=P90, Y=Moda Freq, Anotación P90) ---
        try:
            if len(eventos_riesgo) > 0 and len(perdidas_por_evento) == len(eventos_riesgo) and len(frecuencias_por_evento) == len(eventos_riesgo):

                # 1. Preparar datos para el gráfico (P90, Freq Modo, Importancia actualizada)
                data_riesgos = []
                for idx, evento in enumerate(eventos_riesgo):
                    nombre = evento['nombre']
                    # Usar una copia para evitar modificar los arrays originales si fuera necesario
                    perdidas_evt = np.array(perdidas_por_evento[idx])
                    frecuencias_evt = np.array(frecuencias_por_evento[idx])

                    impacto_medio = np.mean(perdidas_evt) if len(perdidas_evt) > 0 else 0
                    impacto_p90 = np.percentile(perdidas_evt, 90) if len(perdidas_evt) > 0 else 0

                    # --- Calcular Frecuencia MODO (Valor más frecuente) ---
                    if len(frecuencias_evt) > 0:
                        # stats.mode devuelve objeto ModeResult; tomar el primer modo encontrado
                        mode_result = stats.mode(frecuencias_evt, keepdims=True)
                        # Asegurarse de que mode_result.mode no está vacío antes de acceder a [0]
                        # Convertir a float por si acaso, aunque la frecuencia suele ser int
                        frecuencia_modo = float(mode_result.mode[0]) if mode_result.mode.size > 0 else 0
                    else:
                        frecuencia_modo = 0
                    # -------------------------------------------------------

                    # --- Actualizar Importancia usando Frecuencia MODO ---
                    # El tamaño ahora refleja: Impacto Medio * Frecuencia Más Probable
                    importancia = (impacto_medio * frecuencia_modo) + 1e-9 # Epsilon para evitar tamaño cero
                    # ----------------------------------------------------

                    data_riesgos.append({
                        'Nombre': nombre,
                        'ImpactoMedio': impacto_medio,
                        'ImpactoP90': impacto_p90,
                        'FrecuenciaModo': frecuencia_modo, # <-- Usar Modo para el eje Y
                        'Importancia': importancia        # <-- Basada en Modo para el tamaño
                    })

                # 2. Crear DataFrame
                df_riesgos = pd.DataFrame(data_riesgos)

                # Asegurarnos de que no haya valores negativos o cero donde no deben
                df_riesgos['Importancia'] = df_riesgos['Importancia'].clip(lower=1e-9)
                df_riesgos['ImpactoP90'] = df_riesgos['ImpactoP90'].clip(lower=0)
                df_riesgos['ImpactoMedio'] = df_riesgos['ImpactoMedio'].clip(lower=0)
                df_riesgos['FrecuenciaModo'] = df_riesgos['FrecuenciaModo'].clip(lower=0)


                # 3. Generar Gráfico Scatterplot
                fig_mapa = Figure(figsize=(10, 7)) # Tamaño puede requerir ajuste con barra de color
                canvas_mapa = FigureCanvas(fig_mapa)
                # Usar gridspec para controlar mejor la posición de la barra de color
                gs = fig_mapa.add_gridspec(1, 2, width_ratios=[0.9, 0.05], wspace=0.05)
                ax_mapa = fig_mapa.add_subplot(gs[0, 0])
                cax = fig_mapa.add_subplot(gs[0, 1]) # Eje para la barra de color

                # Crear el scatterplot - MODIFICADO: y='FrecuenciaModo', palette='RdYlGn_r', legend=False
                scatter = sns.scatterplot(
                    data=df_riesgos,
                    x='ImpactoMedio',
                    y='FrecuenciaModo',       # <-- MODIFICADO: Eje Y es Modo
                    size='Importancia',     # <-- Tamaño basado en Importancia (Impacto Medio * Freq Modo)
                    hue='ImpactoP90',       # <-- Color sigue siendo Impacto P90
                    palette='RdYlGn_r',     # <-- MODIFICADO: Paleta Verde(bajo)->Rojo(alto)
                    sizes=(50, 1500),       # Rango de tamaño (ajustar)
                    alpha=0.75,             # Transparencia
                    legend=False,           # <-- MODIFICADO: Quitar leyendas automáticas de puntos/tamaño
                    ax=ax_mapa
                )

                # 4. Personalizar Gráfico Principal
                ax_mapa.set_title('Mapa de Riesgos (Impacto Medio vs. Frecuencia Más Probable)', fontsize=14)
                ax_mapa.set_xlabel('Impacto Económico Promedio (Simulado)', fontsize=10)
                ax_mapa.set_ylabel('Frecuencia Anual Más Probable (Modo Simulado)', fontsize=10) # <-- Etiqueta Y actualizada
                ax_mapa.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
                # Usar formato entero para el Modo, ya que representa conteos
                ax_mapa.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{int(y)}'))
                # Ajustar límites para asegurar visibilidad cerca de los ejes
                min_x = df_riesgos['ImpactoMedio'].min() if not df_riesgos.empty else 0
                min_y = df_riesgos['FrecuenciaModo'].min() if not df_riesgos.empty else 0
                ax_mapa.set_xlim(left=max(0, min_x * 0.95))
                ax_mapa.set_ylim(bottom=max(0, min_y * 0.95))
                ax_mapa.grid(True, linestyle='--', alpha=0.5) # Añadir rejilla suave

                # --- MODIFICADO: Añadir anotaciones con Nombre y P90 ---
                n_top_riesgos = 7 # Anotar los N más importantes por tamaño ('Importancia')
                if not df_riesgos.empty:
                    df_riesgos_sorted = df_riesgos.nlargest(min(n_top_riesgos, len(df_riesgos)), 'Importancia')

                    for i, row in df_riesgos_sorted.iterrows():
                        # Crear etiqueta con nombre y P90 formateado en líneas separadas
                        label_texto = f"{row['Nombre']}\nP90: {currency_format(row['ImpactoP90'])}"
                        # Ajustar posición para evitar solapamientos (puede requerir más lógica si es necesario)
                        ax_mapa.text(row['ImpactoMedio'], row['FrecuenciaModo'], # Posición basada en ejes actuales
                                     f"  {label_texto}", # Añadir espacio inicial para separar del punto
                                     fontsize=7, # Reducir tamaño fuente
                                     ha='left', va='center', # Alinear a la izquierda y centrado verticalmente
                                     bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='grey')) # Fondo claro

                # 5. Crear Barra de Color para Leyenda de P90 (hue)
                if not df_riesgos.empty:
                    norm = plt.Normalize(df_riesgos['ImpactoP90'].min(), df_riesgos['ImpactoP90'].max())
                    cmap = plt.get_cmap("RdYlGn_r") # Obtener el colormap
                    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
                    sm.set_array([]) # Necesario

                    # Dibujar la barra de color en el eje 'cax'
                    cbar = fig_mapa.colorbar(sm, cax=cax, orientation='vertical')
                    cbar.set_label('Impacto P90', rotation=270, labelpad=15, fontsize=10)
                    cbar.ax.yaxis.set_major_formatter(FuncFormatter(currency_formatter)) # Formato moneda en la barra
                    cbar.ax.tick_params(labelsize=8) # Tamaño fuente de ticks
                else:
                    # Si no hay datos, ocultar el eje de la barra de color
                     cax.set_visible(False)

                # Ajustar layout general (tight_layout puede no funcionar bien con colorbar añadida así)
                # fig_mapa.tight_layout() # Comentar o eliminar si causa problemas con la barra de color

                # 6. Integrar en PyQt
                tab_mapa = QtWidgets.QWidget()
                layout_mapa = QtWidgets.QVBoxLayout(tab_mapa)
                layout_mapa.addWidget(canvas_mapa)
                # Actualizar nombre de pestaña
                self.graficos_tab_widget.addTab(tab_mapa, "Mapa Riesgos (Modo, P90)")

            else:
                 print("No hay eventos de riesgo o resultados para generar el Mapa de Riesgos.")

        except ImportError:
             # Manejo específico si falta SciPy
             print("Error: Se requiere SciPy para calcular el modo. Instala SciPy: pip install scipy")
             # Mostrar mensaje al usuario en la GUI
             QtWidgets.QMessageBox.warning(self, "Dependencia Faltante",
                                           "Se requiere la librería SciPy para usar la frecuencia modo en el Mapa de Riesgos.\n"
                                           "Por favor, instálala (ej: pip install scipy) y reinicia la aplicación.\n"
                                           "El gráfico no se generará en esta sesión.")
        except Exception as e:
            # Capturar otros errores
            print(f"Error al generar el Mapa de Riesgos (Modo/P90 Anotado): {e}")
            error_message_mapa = traceback.format_exc()
            print(error_message_mapa)
            # QtWidgets.QMessageBox.warning(self, "Error Gráfico", f"No se pudo generar el Mapa de Riesgos (Modo/P90 Anotado):\n{e}")

# --- FIN: Nuevo Gráfico - Mapa de Riesgos (Scatterplot - V3) ---

        # Gráfico de Termómetro de Riesgo
        fig_term = Figure(figsize=(7, 5))
        canvas_term = FigureCanvas(fig_term)
        ax_term = fig_term.add_subplot(111)

        # Definir umbrales de riesgo (puedes ajustarlos según tus criterios)
        perdida_media = np.mean(perdidas_totales)
        umbral_bajo = np.percentile(perdidas_totales, 50)  # Riesgo bajo: hasta el 50%
        umbral_medio = np.percentile(perdidas_totales, 75)  # Riesgo medio: hasta el 75%
        umbral_alto = np.percentile(perdidas_totales, 90)   # Riesgo alto: hasta el 90%
        umbral_critico = np.percentile(perdidas_totales, 95) # Riesgo crítico: más del 95%
        max_valor = np.percentile(perdidas_totales, 99)     # Para la escala máxima

        # Crear el "termómetro" como un gráfico de barras horizontales
        umbrales = [umbral_bajo, umbral_medio - umbral_bajo, umbral_alto - umbral_medio,
                    umbral_critico - umbral_alto, max_valor - umbral_critico]
        colores = ['#90EE90', '#FFFF66', '#FFA500', '#FF6347', '#DC143C']  # Verde claro, amarillo, naranja, rojo claro, rojo oscuro
        etiquetas = ['Bajo', 'Moderado', 'Alto', 'Severo', 'Crítico']

        # Crear barras para los rangos de riesgo
        for i, (umbral, color) in enumerate(zip(umbrales, colores)):
            offset = sum(umbrales[:i])
            ax_term.barh(0, umbral, left=offset, height=0.5, color=color, edgecolor='black', alpha=0.7)

        # Agregar línea vertical para la pérdida media
        ax_term.axvline(x=perdida_media, color='blue', linestyle='-', linewidth=3, label='Pérdida Esperada')

        # Agregar etiquetas de texto para cada zona
        for i, (etiqueta, umbral) in enumerate(zip(etiquetas, umbrales)):
            offset = sum(umbrales[:i]) + umbral/2
            ax_term.text(offset, 0, etiqueta, ha='center', va='center', fontsize=10)

        # Personalizar el gráfico
        ax_term.set_yticks([])  # Eliminar eje Y
        ax_term.set_xlim(0, max_valor * 1.05)  # Agregar un poco de espacio al final
        ax_term.xaxis.set_major_formatter(FuncFormatter(currency_formatter))
        ax_term.set_title('Termómetro de Riesgo: Nivel de Pérdida Esperada', fontsize=12)

        # Agregar flecha señalando la pérdida media
        ax_term.annotate('Pérdida Media', xy=(perdida_media, 0), xytext=(perdida_media, 0.8),
                        arrowprops=dict(facecolor='blue', shrink=0.05, width=2, headwidth=10),
                        ha='center', va='top', fontsize=11, fontweight='bold')

        # Explicación textual
        texto_explicativo = f"""
        • Pérdida Media: {currency_format(perdida_media)}
        • La posición de la flecha azul indica el nivel de riesgo actual
        • Las zonas de color representan diferentes niveles de severidad
        """
        fig_term.text(0.5, 0.05, texto_explicativo, ha='center', va='center', fontsize=9,
                    bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.5'))

        fig_term.tight_layout(rect=[0, 0.15, 1, 0.95])  # Ajustar para dar espacio al texto inferior

        tab13 = QtWidgets.QWidget()
        layout_term = QtWidgets.QVBoxLayout(tab13)
        layout_term.addWidget(canvas_term)
        self.graficos_tab_widget.addTab(tab13, "Nivel de Riesgo")

        # Gráfico de Probabilidad de Impacto tipo Semáforo
        fig_semaforo = Figure(figsize=(8, 6))
        canvas_semaforo = FigureCanvas(fig_semaforo)
        ax_semaforo = fig_semaforo.add_subplot(111)

        # Definir rangos de impacto (ajustar según necesidades del negocio)
        p25 = np.percentile(perdidas_totales, 25)
        p50 = np.percentile(perdidas_totales, 50)
        p75 = np.percentile(perdidas_totales, 75)
        p95 = np.percentile(perdidas_totales, 95)

        rangos = [
            (0, p25, "Impacto\nMuy Bajo"),
            (p25, p50, "Impacto\nBajo"),
            (p50, p75, "Impacto\nModerado"),
            (p75, p95, "Impacto\nAlto"),
            (p95, np.max(perdidas_totales), "Impacto\nCrítico")
        ]

        # Calcular probabilidad para cada rango
        probabilidades = []
        for min_val, max_val, _ in rangos:
            prob = len(perdidas_totales[(perdidas_totales >= min_val) & (perdidas_totales < max_val)]) / len(perdidas_totales)
            probabilidades.append(prob * 100)  # Convertir a porcentaje

        # Colores del semáforo
        colores = ['#90EE90', '#ADFF2F', '#FFFF00', '#FFA500', '#FF4500']  # Verde claro, verde lima, amarillo, naranja, rojo

        # Crear gráfico de barras
        bars = ax_semaforo.barh([r[2] for r in rangos], probabilidades, color=colores, edgecolor='black')

        # Agregar etiquetas de porcentaje dentro de las barras
        for bar, porcentaje in zip(bars, probabilidades):
            width = bar.get_width()
            if width >= 10:  # Solo agregar texto si hay suficiente espacio
                ax_semaforo.text(width / 2, bar.get_y() + bar.get_height() / 2,
                              f"{porcentaje:.1f}%",
                              ha='center', va='center', fontweight='bold')

        # Personalizar gráfico
        ax_semaforo.set_xlabel('Probabilidad (%)')
        ax_semaforo.set_title('Probabilidad de Diferentes Niveles de Impacto', fontsize=14)
        ax_semaforo.set_xlim(0, 100)
        ax_semaforo.grid(axis='x', linestyle='--', alpha=0.7)

        # Añadir valores monetarios para cada nivel
        for i, (min_val, max_val, nombre) in enumerate(rangos):
            ax_semaforo.text(105, i, f"{currency_format(min_val)} - {currency_format(max_val)}",
                          va='center', ha='left', fontsize=9)

        # Añadir leyenda explicativa
        leyenda = """
        Este gráfico muestra la probabilidad de que ocurran pérdidas
        en diferentes rangos de severidad. Cuanto más a la derecha,
        mayor es la probabilidad de ese nivel de impacto.

        Los colores indican la gravedad del impacto:
        • Verde: Impacto manejable
        • Amarillo: Requiere atención
        • Naranja/Rojo: Impacto severo que requiere acción inmediata
        """
        fig_semaforo.text(0.5, 0.02, leyenda, ha='center', va='bottom', fontsize=9,
                        bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round', alpha=0.9))

        fig_semaforo.tight_layout(rect=[0, 0.15, 0.85, 0.95])  # Ajustar para texto explicativo

        tab14 = QtWidgets.QWidget()
        layout_semaforo = QtWidgets.QVBoxLayout(tab14)
        layout_semaforo.addWidget(canvas_semaforo)
        self.graficos_tab_widget.addTab(tab14, "Semáforo de Impacto")

        # Gráfico "¿Qué pasaría si...?" - Comparativa de escenarios con impacto en negocio
        fig_escenarios = Figure(figsize=(8, 6))
        canvas_escenarios = FigureCanvas(fig_escenarios)
        ax_escenarios = fig_escenarios.add_subplot(111)

        # Obtener pérdidas para diferentes percentiles
        escenarios = [
            ("Caso típico\n(Promedio)", np.mean(perdidas_totales)),
            ("Caso adverso\n(1 de cada 10 veces)", np.percentile(perdidas_totales, 90)),
            ("Caso muy adverso\n(1 de cada 20 veces)", np.percentile(perdidas_totales, 95)),
            ("Caso extremo\n(1 de cada 100 veces)", np.percentile(perdidas_totales, 99))
        ]

        # Crear gráfico de barras para escenarios
        nombres = [e[0] for e in escenarios]
        valores = [e[1] for e in escenarios]

        bars = ax_escenarios.bar(nombres, valores, color=['#7DCEA0', '#F8C471', '#E59866', '#EC7063'])

        # Añadir etiquetas de valor encima de cada barra
        for bar in bars:
            height = bar.get_height()
            ax_escenarios.text(bar.get_x() + bar.get_width()/2., height + valores[-1]*0.02,
                            f'{currency_format(height)}',
                            ha='center', va='bottom', rotation=0, fontsize=10)

        # Personalizar gráfico
        ax_escenarios.set_ylabel('Impacto Económico Potencial')
        ax_escenarios.set_title('¿Qué impacto habría si se materializara el riesgo?', fontsize=14)
        ax_escenarios.set_ylim(0, valores[-1] * 1.3)  # Dar espacio para las etiquetas
        ax_escenarios.yaxis.set_major_formatter(FuncFormatter(currency_formatter))

        # Añadir líneas de referencia con contexto de negocio (ajustar según el caso específico)
        referencias = [
            (valores[0] * 0.5, "Limite dentro de la Tolerancia", "green"),
            (valores[1] * 0.9, "Máximo de Impacto Aceptable", "orange"),
            (valores[2] * 1.05, "Punto Crítico", "red")
        ]

        for valor, nombre, color in referencias:
            ax_escenarios.axhline(y=valor, color=color, linestyle='--', alpha=0.7)
            ax_escenarios.text(len(escenarios) - 0.5, valor, f"  {nombre}: {currency_format(valor)}",
                            va='center', ha='right', color=color, fontweight='bold', fontsize=9)

        # Añadir explicación
        explicacion = """
        Este gráfico muestra el impacto económico en diferentes escenarios:
        • Caso típico: El resultado que se podría esperar en circunstancias normales
        • Caso adverso: Un resultado desfavorable que podría ocurrir ocasionalmente (10% de probabilidad)
        • Caso muy adverso: Un resultado seriamente desfavorable pero posible (5% de probabilidad)
        • Caso extremo: Un resultado excepcionalmente grave pero plausible (1% de probabilidad)

        Las líneas horizontales representan umbrales importantes para el negocio.
        """
        fig_escenarios.text(0.5, 0.01, explicacion, ha='center', va='bottom', fontsize=9,
                          bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round', alpha=0.9))

        fig_escenarios.tight_layout(rect=[0, 0.2, 1, 0.95])  # Ajustar para texto explicativo

        tab15 = QtWidgets.QWidget()
        layout_escenarios = QtWidgets.QVBoxLayout(tab15)
        layout_escenarios.addWidget(canvas_escenarios)
        self.graficos_tab_widget.addTab(tab15, "Escenarios")

        # Gráfico de Traducción del Riesgo a Términos de Negocio
        fig_traduccion = Figure(figsize=(9, 6))
        canvas_traduccion = FigureCanvas(fig_traduccion)
        ax_traduccion = fig_traduccion.add_subplot(111)

        # Definir equivalencias de negocio (ajustar según la organización)
        media_perdidas = np.mean(perdidas_totales)
        p99_perdidas = np.percentile(perdidas_totales, 99)

        # Crear datos para el gráfico - ajusta estos valores según tu contexto
        impacto_unidades = {
            "% del Presupuesto Anual": [media_perdidas / 1000000 * 100, p99_perdidas / 1000000 * 100],
            "Meses de Ingresos": [media_perdidas / 100000, p99_perdidas / 100000],
            "GMV Anual": [media_perdidas / 50000, p99_perdidas / 50000],
            "EBITDA": [media_perdidas / 200000, p99_perdidas / 200000],
            "Días de operación\ndetenida": [media_perdidas / 30000, p99_perdidas / 30000]
        }

        # Preparar datos para el gráfico
        categorias = list(impacto_unidades.keys())
        impacto_media = [val[0] for val in impacto_unidades.values()]
        impacto_extremo = [val[1] for val in impacto_unidades.values()]

        # Posición de las barras
        x = np.arange(len(categorias))
        ancho = 0.35

        # Crear barras para casos medio y extremo
        rects1 = ax_traduccion.barh(x - ancho/2, impacto_media, ancho, color='#5DADE2', label='Caso promedio')
        rects2 = ax_traduccion.barh(x + ancho/2, impacto_extremo, ancho, color='#EC7063', label='Caso extremo (P99)')

        # Añadir etiquetas y título
        ax_traduccion.set_title('El Impacto del Riesgo en Términos de Negocio', fontsize=14)
        ax_traduccion.set_xlabel('Magnitud del Impacto')
        ax_traduccion.set_yticks(x)
        ax_traduccion.set_yticklabels(categorias)
        ax_traduccion.legend(loc='upper right')

        # Añadir valores dentro o al final de las barras
        for i, rect in enumerate(rects1):
            width = rect.get_width()
            ax_traduccion.text(max(width + 0.5, 1), rect.get_y() + rect.get_height()/2.,
                            f'{width:.1f}', ha='left', va='center')

        for i, rect in enumerate(rects2):
            width = rect.get_width()
            ax_traduccion.text(max(width + 0.5, 1), rect.get_y() + rect.get_height()/2.,
                            f'{width:.1f}', ha='left', va='center')

        # Añadir explicación
        explicacion = """
        Este gráfico traduce el impacto del riesgo a términos que son relevantes para el negocio.
        Compara el caso promedio (azul) con un escenario extremo que ocurriría aproximadamente una vez cada 100 eventos (rojo).

        Nota: Estos valores son aproximaciones basadas en los parámetros actuales de la organización.
        """
        fig_traduccion.text(0.5, 0.01, explicacion, ha='center', va='bottom', fontsize=9,
                          bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round', alpha=0.9))

        fig_traduccion.tight_layout(rect=[0, 0.1, 0.95, 0.95])  # Ajustar para texto explicativo

        tab16 = QtWidgets.QWidget()
        layout_traduccion = QtWidgets.QVBoxLayout(tab16)
        layout_traduccion.addWidget(canvas_traduccion)
        self.graficos_tab_widget.addTab(tab16, "Impacto en Negocio")

        # Gráfico de Calendario de Riesgo
        fig_calendario = Figure(figsize=(8, 6))
        canvas_calendario = FigureCanvas(fig_calendario)
        ax_calendario = fig_calendario.add_subplot(111)

        # Calcular frecuencia promedio de eventos
        eventos_por_año = np.mean(frecuencias_totales)  # Suponiendo que la simulación es anual

        # Definir niveles de pérdida y calcular sus frecuencias
        niveles_perdida = [
            (np.percentile(perdidas_totales, 50), "Pérdida Moderada"),
            (np.percentile(perdidas_totales, 75), "Pérdida Significativa"),
            (np.percentile(perdidas_totales, 90), "Pérdida Grave"),
            (np.percentile(perdidas_totales, 95), "Pérdida Muy Grave"),
            (np.percentile(perdidas_totales, 99), "Pérdida Catastrófica")
        ]

        # Calcular frecuencias (convertir a periodos de tiempo más intuitivos)
        periodos = []
        for umbral, nombre in niveles_perdida:
            # Probabilidad de que una pérdida exceda el umbral
            prob_exceder = len(perdidas_totales[perdidas_totales > umbral]) / len(perdidas_totales)

            # Frecuencia: cada cuántos años en promedio se espera un evento que exceda este umbral
            if prob_exceder > 0:
                frecuencia_años = 1 / (prob_exceder * eventos_por_año)
            else:
                frecuencia_años = float('inf')  # Nunca sucede en nuestra simulación

            periodos.append((frecuencia_años, nombre, umbral))

        # Ordenar por frecuencia (de más frecuente a menos frecuente)
        periodos.sort()

        # Crear escala de tiempo más intuitiva
        etiquetas_tiempo = []
        valores_tiempo = []
        nombres = []
        valores_perdida = []

        for frecuencia_años, nombre, umbral in periodos:
            nombres.append(nombre)
            valores_perdida.append(umbral)

            # Convertir a una escala de tiempo más comprensible
            if frecuencia_años < 1/12:  # Menos de un mes
                dias = frecuencia_años * 365
                etiquetas_tiempo.append(f"Cada {dias:.1f} días")
                valores_tiempo.append(dias/365)  # Convertir a fracción de año para el gráfico
            elif frecuencia_años < 1:  # Menos de un año
                meses = frecuencia_años * 12
                etiquetas_tiempo.append(f"Cada {meses:.1f} meses")
                valores_tiempo.append(frecuencia_años)
            elif frecuencia_años < float('inf'):
                etiquetas_tiempo.append(f"Cada {frecuencia_años:.1f} años")
                valores_tiempo.append(frecuencia_años)
            else:
                etiquetas_tiempo.append("Extremadamente raro")
                valores_tiempo.append(100)  # Valor arbitrario alto para el gráfico

        # Colores según la gravedad
        colores = ['#7DCEA0', '#F8C471', '#F5B041', '#EC7063', '#C0392B']

        # Crear gráfico de barras horizontales
        bars = ax_calendario.barh(nombres, valores_tiempo, color=colores)

        # Añadir etiquetas de tiempo
        for i, (bar, etiqueta) in enumerate(zip(bars, etiquetas_tiempo)):
            ax_calendario.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                            etiqueta, va='center', ha='left')

        # Personalizar gráfico
        ax_calendario.set_title('¿Con qué frecuencia podemos esperar estos niveles de pérdida?', fontsize=14)
        ax_calendario.set_xlabel('Frecuencia (años)')  # Eje base en años
        ax_calendario.set_xscale('log')  # Escala logarítmica para mejor visualización
        ax_calendario.set_xlim(min(valores_tiempo) * 0.5, max(valores_tiempo) * 2)
        ax_calendario.grid(axis='x', which='both', linestyle='--', alpha=0.7)

        # Añadir etiquetas de valor monetario
        for i, (nombre, valor) in enumerate(zip(nombres, valores_perdida)):
            ax_calendario.text(ax_calendario.get_xlim()[0] * 1.5, i,
                            f"{currency_format(valor)}", va='center', ha='left', fontweight='bold')

        # Añadir explicación
        explicacion = """
        Este "calendario de riesgo" muestra la frecuencia esperada con la que podrían ocurrir diferentes niveles de pérdida.
        Las pérdidas más pequeñas ocurren más frecuentemente, mientras que las pérdidas catastróficas son mucho más raras.

        Los valores monetarios en el gráfico indican el umbral mínimo de pérdida para cada categoría.
        """
        fig_calendario.text(0.5, 0.01, explicacion, ha='center', va='bottom', fontsize=9,
                          bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round', alpha=0.9))

        fig_calendario.tight_layout(rect=[0, 0.15, 1, 0.95])  # Ajustar para texto explicativo

        tab17 = QtWidgets.QWidget()
        layout_calendario = QtWidgets.QVBoxLayout(tab17)
        layout_calendario.addWidget(canvas_calendario)
        self.graficos_tab_widget.addTab(tab17, "Calendario de Riesgo")

# --- Inicio: Código para Análisis de Sensibilidad SHAP ---
        try:
            print("Iniciando análisis SHAP...") # Mensaje para depuración

            # 1. Preparar los datos que SHAP analizará
            #    Usaremos la pérdida promedio de cada evento como "característica"
            #    y la pérdida promedio total como el "resultado" a explicar.
            nombres_eventos_shap = [evento['nombre'] for evento in eventos_riesgo]
            perdidas_medias_por_evento = [np.mean(p_ev) for p_ev in perdidas_por_evento]

            # Creamos una tabla (DataFrame) con una sola fila: las pérdidas medias por evento
            X_shap = pd.DataFrame([perdidas_medias_por_evento], columns=nombres_eventos_shap)

            # El resultado que queremos explicar es la pérdida media total
            resultado_a_explicar = np.mean(perdidas_totales)
            print(f"Resultado a explicar (Pérdida Media Total): {resultado_a_explicar}")
            print("Datos de entrada para SHAP (Pérdidas Medias por Evento):\n", X_shap)


            # 2. Crear un modelo simple para que SHAP lo explique
            #    Como estamos explicando cómo las partes (medias por evento)
            #    suman al todo (media total), un modelo lineal es adecuado aquí.
            modelo_simple = sklearn.linear_model.LinearRegression()
            # "Entrenamos" el modelo (aunque con una sola fila, solo captura la relación)
            # Necesitamos al menos dos puntos para entrenar, duplicamos la fila con ruido mínimo.
            if len(X_shap) == 1:
                 X_train_shap = pd.concat([X_shap, X_shap * (1 + np.random.randn(*X_shap.shape)*1e-9)], ignore_index=True)
                 y_train_shap = np.array([resultado_a_explicar, resultado_a_explicar * (1 + np.random.randn()*1e-9)])
            else: # Si por alguna razón hubiera más filas (no debería en este caso)
                 X_train_shap = X_shap
                 y_train_shap = np.full(len(X_shap), resultado_a_explicar) # Simplificado

            modelo_simple.fit(X_train_shap, y_train_shap)


            # 3. Usar SHAP para explicar el modelo
            #    Le decimos a SHAP que use nuestro modelo simple y nuestros datos
            explainer = shap.Explainer(modelo_simple, X_train_shap)
            shap_values = explainer(X_shap) # Explicamos la predicción para nuestra única fila de datos reales

            print(f"Valor base SHAP (E[f(X)]): {shap_values.base_values[0]}")
            print(f"Valores SHAP calculados: {shap_values.values[0]}")
            print(f"Suma(Base + SHAP values): {shap_values.base_values[0] + np.sum(shap_values.values[0])}")


            # 4. Crear el gráfico SHAP (Waterfall)
            #    Este gráfico muestra cómo cada evento "empuja" el resultado
            #    desde un valor base hasta el valor final.

            # Creamos una figura de Matplotlib explícitamente para SHAP
            fig_shap = plt.figure(figsize=(10, 9)) # Ajusta tamaño si es necesario
            ax_shap = fig_shap.add_subplot(111)

            # Generamos el gráfico waterfall en nuestros ejes
            # Usamos el índice 0 porque solo tenemos una fila/predicción que explicar
            shap.plots.waterfall(shap_values[0], max_display=15, show=False)

            # Ajustes adicionales al gráfico generado por SHAP
            plt.title("¿Cómo Contribuye Cada Riesgo a la Pérdida Promedio TOTAL?", fontsize=14)
            plt.xlabel(f"Contribución a la Pérdida Promedio ({currency_format(resultado_a_explicar)})", fontsize=10)
            ax_shap = plt.gca()
            ax_shap.xaxis.set_major_formatter(FuncFormatter(currency_formatter)) # Asegurar formato moneda
            # Quitar etiqueta del eje Y si la hubiera (a veces SHAP añade 'Features')
            ax_shap.set_ylabel('')
            # Ajustar márgenes para mejor visualización
            plt.tight_layout()

            # 5. Incrustar el gráfico SHAP en una nueva pestaña de PyQt
            canvas_shap = FigureCanvas(fig_shap)
            tab_shap = QtWidgets.QWidget()
            layout_shap = QtWidgets.QVBoxLayout(tab_shap)
            layout_shap.addWidget(canvas_shap) # Añadir el gráfico
            self.graficos_tab_widget.addTab(tab_shap, "Análisis SHAP")

            # --- AÑADIR TEXTO EXPLICATIVO ---
            texto_explicacion_shap = """
            <b>¿Cómo leer este gráfico?</b>
            <ul>
                <li><b>Punto de Partida (E[f(X)]):</b> Es la 'pérdida promedio base' antes de considerar los riesgos individuales.</li>
                <li><b>Barras Rojas:</b> Muestran cuánto 'aumenta' la pérdida promedio cada evento de riesgo listado.</li>
                <li><b>Barras Azules (si hubiera):</b> Mostrarían cuánto 'disminuye' la pérdida promedio cada evento.</li>
                <li><b>Largo de la Barra:</b> Indica la 'fuerza' o importancia de esa contribución (aumento/disminución).</li>
                <li><b>Punto Final (f(x)):</b> Es la 'pérdida promedio total' calculada después de sumar todas las contribuciones al punto de partida. Coincide con la media mostrada arriba.</li>
                <li><i>Nota: Si aparece 'Features n#', agrupa la contribución de los n riesgos restantes menos importantes.</i></li>
            </ul>
            """
            label_explicacion_shap = QtWidgets.QLabel(texto_explicacion_shap)
            label_explicacion_shap.setWordWrap(True) # Para que el texto se ajuste
            label_explicacion_shap.setStyleSheet("font-size: 11px; background-color: #f0f0f0; border: 1px solid #ccc; padding: 5px; border-radius: 3px;") # Estilo opcional
            layout_shap.addWidget(label_explicacion_shap) # Añadir el texto debajo del gráfico
            # --- FIN TEXTO EXPLICATIVO ---

            self.graficos_tab_widget.addTab(tab_shap, "Contribución SHAP (Media)") # Nombre de pestaña más claro

            plt.close(fig_shap)
            print("Análisis SHAP completado y gráfico añadido.")

        except Exception as e:
            # Si algo falla en el análisis SHAP, muestra un mensaje pero no detiene la app
            print(f"Error durante el análisis SHAP: {e}")
            error_message_shap = traceback.format_exc()
            print(error_message_shap)
            # Opcional: Mostrar un mensaje de error al usuario en la GUI si lo deseas
            # QtWidgets.QMessageBox.warning(self, "Error SHAP", f"No se pudo generar el gráfico SHAP:\n{e}")

        # --- Fin: Código para Análisis de Sensibilidad SHAP ---

# --- Inicio: Código para Análisis SHAP (Contribución a Pérdida Media > P90) ---
        try:
            print("Iniciando análisis SHAP para la cola P90...") # Mensaje para seguimiento

            # 1. Identificar las simulaciones en la cola (superiores al P90)
            percentil_90_valor = np.percentile(perdidas_totales, 90)
            indices_cola_p90 = np.where(perdidas_totales > percentil_90_valor)[0]

            # 2. Verificar si hay simulaciones en la cola
            if len(indices_cola_p90) == 0:
                print("Advertencia: No hay simulaciones por encima del P90. Omitiendo análisis SHAP de cola P90.")
                # Si no hay datos en la cola, no podemos continuar con este análisis específico.
            else:
                print(f"Se encontraron {len(indices_cola_p90)} simulaciones por encima del P90 ({currency_format(percentil_90_valor)}).")

                # 3. Calcular promedios DENTRO de la cola P90
                #    Promedio de la pérdida total SOLO en esas simulaciones de cola
                perdida_media_cola_p90 = np.mean(perdidas_totales[indices_cola_p90])
                #    Promedio de la pérdida de CADA evento SOLO en esas simulaciones de cola
                perdidas_medias_evento_cola_p90 = [np.mean(p_ev[indices_cola_p90]) for p_ev in perdidas_por_evento]

                # 4. Preparar datos para SHAP (cola P90)
                #    Los nombres de los eventos son los mismos
                nombres_eventos_shap_cola = [evento['nombre'] for evento in eventos_riesgo]
                #    Creamos la tabla (DataFrame) con una sola fila: las pérdidas medias DENTRO DE LA COLA por evento
                X_shap_cola = pd.DataFrame([perdidas_medias_evento_cola_p90], columns=nombres_eventos_shap_cola)

                print(f"Resultado a explicar (Pérdida Media > P90): {currency_format(perdida_media_cola_p90)}")
                print("Datos de entrada para SHAP (Pérdidas Medias > P90 por Evento):\n", X_shap_cola)

                # 5. Crear y ajustar modelo lineal simple para la cola P90
                #    Usamos el mismo truco de duplicar con ruido mínimo para poder ajustar el modelo
                modelo_simple_cola = sklearn.linear_model.LinearRegression()
                X_train_shap_cola = pd.concat([X_shap_cola, X_shap_cola * (1 + np.random.randn(*X_shap_cola.shape)*1e-9)], ignore_index=True)
                y_train_shap_cola = np.array([perdida_media_cola_p90, perdida_media_cola_p90 * (1 + np.random.randn()*1e-9)])
                modelo_simple_cola.fit(X_train_shap_cola, y_train_shap_cola)

                # 6. Usar SHAP para explicar el modelo de la cola P90
                #    Creamos un nuevo explicador para este modelo y datos de cola
                explainer_cola = shap.Explainer(modelo_simple_cola, X_train_shap_cola)
                shap_values_cola = explainer_cola(X_shap_cola) # Explicamos la predicción para los datos reales de la cola

                print(f"Valor base SHAP (Cola P90): {shap_values_cola.base_values[0]}")
                print(f"Valores SHAP calculados (Cola P90): {shap_values_cola.values[0]}")

                # 7. Crear el gráfico SHAP Waterfall para la cola P90
                #    Creamos una NUEVA figura y ejes para este gráfico
                fig_shap_cola = plt.figure(figsize=(10, 9)) # Un poco más alto
                ax_shap_cola = fig_shap_cola.add_subplot(111)

                # Generamos el gráfico waterfall en los nuevos ejes
                shap.plots.waterfall(shap_values_cola[0], max_display=15, show=False)

                # Ajustes al gráfico específico de la cola P90
                plt.title(f"¿Qué Riesgos Impulsan Más la Pérdida en Escenarios Malos (>P90)?", fontsize=14)
                plt.xlabel(f"Contribución a la Pérdida Promedio Condicional (>P90 = {currency_format(percentil_90_valor)})", fontsize=10)
                ax_shap_cola = plt.gca()
                ax_shap_cola.xaxis.set_major_formatter(FuncFormatter(currency_formatter)) # Asegurar formato moneda
                ax_shap_cola.set_ylabel('') # Quitar etiqueta Y
                plt.tight_layout()

                # 8. Incrustar el gráfico SHAP (Cola P90) en una NUEVA pestaña PyQt
                canvas_shap_cola = FigureCanvas(fig_shap_cola)
                tab_shap_cola = QtWidgets.QWidget()
                # Layout vertical para gráfico y texto
                layout_shap_cola = QtWidgets.QVBoxLayout(tab_shap_cola)
                layout_shap_cola.addWidget(canvas_shap_cola) # Añadir gráfico
                # ¡Asegúrate de usar un nombre diferente para la pestaña!
                # --- AÑADIR TEXTO EXPLICATIVO ---
                texto_explicacion_shap_cola = f"""
                <b>¿Cómo leer este gráfico? (Enfoque en Pérdidas Altas > {currency_format(percentil_90_valor)})</b>
                <ul>
                    <li><b>Punto de Partida (E[f(X)]):</b> Es la 'pérdida promedio' esperada <i>solo en los peores escenarios</i> (aquellos sobre P90).</li>
                    <li><b>Barras Rojas:</b> Muestran cuánto 'aumenta' la pérdida (ya alta) cada evento de riesgo en estos malos escenarios.</li>
                    <li><b>Barras Azules (si hubiera):</b> Mostrarían cuánto 'disminuye' la pérdida cada evento en estos malos escenarios.</li>
                    <li><b>Largo de la Barra:</b> Indica la 'fuerza' o importancia de esa contribución en los <i>escenarios de cola</i>.</li>
                    <li><b>Punto Final (f(x)):</b> Es la 'pérdida promedio' calculada <i>solo en los peores escenarios</i>, después de sumar todas las contribuciones.</li>
                    <li><i>Nota: Se enfoca en entender qué riesgos son más críticos cuando las pérdidas totales ya son significativas.</i></li>
                </ul>
                """
                label_explicacion_shap_cola = QtWidgets.QLabel(texto_explicacion_shap_cola)
                label_explicacion_shap_cola.setWordWrap(True)
                label_explicacion_shap_cola.setStyleSheet("font-size: 11px; background-color: #f0f0f0; border: 1px solid #ccc; padding: 5px; border-radius: 3px;") # Estilo opcional
                layout_shap_cola.addWidget(label_explicacion_shap_cola) # Añadir texto debajo
                # --- FIN TEXTO EXPLICATIVO ---

                self.graficos_tab_widget.addTab(tab_shap_cola, "Contribución SHAP (>P90)") # Nombre de pestaña más claro

                plt.close(fig_shap_cola)
                print("Análisis SHAP (Cola P90) completado y gráfico añadido.")

        except Exception as e:
            # Capturar errores específicos de este segundo análisis SHAP
            print(f"Error durante el análisis SHAP de cola P90: {e}")
            error_message_shap_cola = traceback.format_exc()
            print(error_message_shap_cola)
            # Opcional: Mostrar advertencia al usuario
            # QtWidgets.QMessageBox.warning(self, "Error SHAP (Cola)", f"No se pudo generar el gráfico SHAP para la cola P90:\n{e}")

        # --- Fin: Código para Análisis SHAP (Contribución a Pérdida Media > P90) ---

    def exportar_a_pdf(self):
        # Verificar si hay resultados para exportar
        if not hasattr(self, 'resultados_simulacion'):
            QtWidgets.QMessageBox.warning(self, "Advertencia",
                                          "No hay resultados de simulación para exportar. Por favor, ejecute una simulación primero.")
            return

        # Solicitar la ruta donde guardar el PDF
        pdf_filename = self.solicitar_ruta_guardado_pdf()
        if not pdf_filename:
            return  # Usuario canceló la selección de archivo

        try:
            # Aquí generamos el PDF con los últimos resultados de simulación
            perdidas_totales = self.resultados_simulacion.get('perdidas_totales')
            frecuencias_totales = self.resultados_simulacion.get('frecuencias_totales')
            perdidas_por_evento = self.resultados_simulacion.get('perdidas_por_evento')
            frecuencias_por_evento = self.resultados_simulacion.get('frecuencias_por_evento')
            eventos_riesgo = self.resultados_simulacion.get('eventos_riesgo')

            if not all([perdidas_totales is not None, frecuencias_totales is not None,
                      perdidas_por_evento is not None, frecuencias_por_evento is not None,
                      eventos_riesgo is not None]):
                QtWidgets.QMessageBox.warning(self, "Advertencia",
                                            "Los datos de la simulación están incompletos. Por favor, ejecute la simulación nuevamente.")
                return

            # Crear el PDF usando ReportLab
            c = canvas.Canvas(pdf_filename, pagesize=A4)
            ancho_pagina, alto_pagina = A4
            # Establecer un margen
            margen = 2 * cm
            ancho_texto = ancho_pagina - 2 * margen
            alto_texto = alto_pagina - 2 * margen

            # Generar el texto de los resultados
            texto_resultados = self.generar_texto_resultados(
                perdidas_totales, frecuencias_totales,
                perdidas_por_evento, frecuencias_por_evento, eventos_riesgo
            )

            # Crear un objeto de texto
            texto_page = c.beginText(margen, alto_pagina - margen)
            texto_page.setFont("Helvetica", 10)

            # Dividir el texto en líneas y agregarlo al PDF
            lineas = texto_resultados.split('\n')
            for linea in lineas:
                texto_page.textLine(linea)
                if texto_page.getY() < margen:
                    # Añadir la página de texto y comenzar una nueva
                    c.drawText(texto_page)
                    c.showPage()
                    texto_page = c.beginText(margen, alto_pagina - margen)
                    texto_page.setFont("Helvetica", 10)
            c.drawText(texto_page)
            c.showPage()

            # Generar los gráficos y agregarlos al PDF
            figuras = self.generar_figuras(perdidas_totales, frecuencias_totales,
                                          perdidas_por_evento, frecuencias_por_evento, eventos_riesgo)

            for idx, figura in enumerate(figuras):
                # Guardar la figura en memoria en lugar de en un archivo
                img_data = io.BytesIO()
                figura.savefig(img_data, format='png', dpi=300, bbox_inches='tight')
                img_data.seek(0)  # Restablece el cursor al inicio del archivo en memoria

                # Obtener el tamaño de la imagen
                img_width, img_height = figura.get_size_inches()
                img_width *= 300  # Convertir tamaño a píxeles (dpi)
                img_height *= 300

                # Calcular el escalado para ajustar la imagen al tamaño de la página
                ratio = min((ancho_texto) / img_width, (alto_texto) / img_height)
                scaled_width = img_width * ratio
                scaled_height = img_height * ratio

                c.drawImage(
                    ImageReader(img_data),
                    margen,
                    (alto_pagina - margen - scaled_height),
                    width=scaled_width,
                    height=scaled_height,
                    preserveAspectRatio=True,
                    anchor='c'
                )
                c.showPage()  # Añadir una nueva página para la siguiente imagen

            c.save()

            QtWidgets.QMessageBox.information(self, "Reporte PDF",
                                            f"El reporte ha sido guardado exitosamente en {pdf_filename}.")
        except Exception as e:
            error_message = traceback.format_exc()
            QtWidgets.QMessageBox.critical(self, "Error",
                                          f"No se pudo generar el reporte PDF:\n{error_message}")

    def generar_texto_resultados(self, perdidas_totales, frecuencias_totales, perdidas_por_evento, frecuencias_por_evento, eventos_riesgo):
        """Calcula estadísticas y genera texto de resultados."""
        texto_resultados = ""

        # Calculamos estadísticas para pérdidas agregadas
        percentiles_valores = [10, 20, 30, 40, 50, 60, 70, 75, 80, 85, 90, 92, 95, 99, 99.99]
        percentiles = np.percentile(perdidas_totales, percentiles_valores)
        media = np.mean(perdidas_totales)
        desviacion_estandar = np.std(perdidas_totales)

        # Estadísticas de frecuencias agregadas
        min_freq_total = int(frecuencias_totales.min())
        max_freq_total = int(frecuencias_totales.max())
        mode_freq_total = int(stats.mode(frecuencias_totales, keepdims=True).mode[0]) if len(frecuencias_totales) > 0 else 0

        # Creamos DataFrame de percentiles para pérdidas agregadas
        percentiles_df = pd.DataFrame({
            'Percentil (%)': percentiles_valores,
            'Valor de Pérdida': percentiles
        })

        # Formateamos los valores de pérdida sin decimales
        percentiles_df['Valor de Pérdida'] = percentiles_df['Valor de Pérdida'].round(0).astype(int)
        percentiles_df['Valor de Pérdida'] = percentiles_df['Valor de Pérdida'].apply(currency_format)

        # Formateamos los percentiles (%), dejando decimales donde corresponda
        percentiles_df['Percentil (%)'] = percentiles_df['Percentil (%)'].apply(
            lambda x: ('{:.2f}'.format(x).replace('.', ',')) if x != int(x) else ('{:.0f}'.format(x)))

        # Obtener resumen ejecutivo
        var_90 = np.percentile(perdidas_totales, 90)
        opvar_99 = np.percentile(perdidas_totales, 99)
        opvar = perdidas_totales[perdidas_totales >= opvar_99].mean()
        texto_resultados += obtener_resumen_ejecutivo_texto(media, desviacion_estandar, var_90, opvar_99, opvar,
                                                            min_freq_total, mode_freq_total, max_freq_total)

        # Obtener tabla de percentiles
        texto_resultados += obtener_tabla_percentiles_texto(percentiles_df, "Percentiles de Pérdida Agregada")

        # Agregamos cálculo y muestra de estadísticas para cada evento de riesgo
        for idx, (perdidas_evento, frecuencias_evento) in enumerate(zip(perdidas_por_evento, frecuencias_por_evento)):
            nombre_evento = eventos_riesgo[idx]['nombre']
            media_evento = np.mean(perdidas_evento)
            desviacion_evento = np.std(perdidas_evento)
            percentiles_evento = np.percentile(perdidas_evento, percentiles_valores)
            min_freq = int(frecuencias_evento.min())
            max_freq = int(frecuencias_evento.max())
            mode_freq = int(stats.mode(frecuencias_evento, keepdims=True).mode[0]) if len(frecuencias_evento) > 0 else 0
            percentiles_evento_df = pd.DataFrame({
                'Percentil (%)': percentiles_valores,
                'Valor de Pérdida': percentiles_evento
            })
            percentiles_evento_df['Valor de Pérdida'] = percentiles_evento_df['Valor de Pérdida'].round(0).astype(int)
            percentiles_evento_df['Valor de Pérdida'] = percentiles_evento_df['Valor de Pérdida'].apply(currency_format)
            percentiles_evento_df['Percentil (%)'] = percentiles_evento_df['Percentil (%)'].apply(
                lambda x: ('{:.2f}'.format(x).replace('.', ',')) if x != int(x) else ('{:.0f}'.format(x)))

            texto_resultados += f"\nEstadísticas para el Evento de Riesgo: {nombre_evento}\n"
            texto_resultados += f"Media de Impacto: {currency_format(round(media_evento))}\n"
            texto_resultados += f"Desviación Estándar: {currency_format(round(desviacion_evento))}\n"
            texto_resultados += f"Número mínimo de eventos materializados: {min_freq}\n"
            texto_resultados += f"Número más probable de eventos materializados: {mode_freq}\n"
            texto_resultados += f"Número máximo de eventos materializados: {max_freq}\n"
            texto_resultados += "Percentiles de Pérdida:\n"
            texto_resultados += tabulate(percentiles_evento_df, headers='keys', tablefmt='fancy_grid', showindex=False)
            texto_resultados += "\n"

        return texto_resultados

    def guardar_configuracion(self):
        if not self.eventos_riesgo and not self.scenarios:
            QtWidgets.QMessageBox.warning(self, "Advertencia", "No hay datos para guardar.")
            return
        options = QtWidgets.QFileDialog.Options()
        filepath, _ = QtWidgets.QFileDialog.getSaveFileName(self, "Guardar Simulación", "",
                                                            "JSON Files (*.json);;All Files (*)", options=options)
        if filepath:
            try:
                configuracion = {
                    'num_simulaciones': int(self.num_simulaciones_var.text()),
                    'eventos_riesgo': [],
                    'scenarios': [],
                    'current_scenario_name': self.current_scenario.nombre if self.current_scenario else None
                }
                # Guardar eventos de riesgo
                for evento in self.eventos_riesgo:
                    evento_data = evento.copy()
                    # Removemos objetos no serializables
                    if 'dist_severidad' in evento_data: # Comprobar si existe antes de borrar
                        del evento_data['dist_severidad']
                    if 'dist_frecuencia' in evento_data: # Comprobar si existe antes de borrar
                        del evento_data['dist_frecuencia']
                    configuracion['eventos_riesgo'].append(evento_data)

                # Guardar escenarios
                for scenario in self.scenarios:
                    escenario_data = {
                        'nombre': scenario.nombre,
                        'descripcion': scenario.descripcion,
                        'eventos_riesgo': []
                    }
                    for evento in scenario.eventos_riesgo:
                        evento_data = evento.copy()
                        # Removemos objetos no serializables
                        del evento_data['dist_severidad']
                        del evento_data['dist_frecuencia']
                        escenario_data['eventos_riesgo'].append(evento_data)
                    configuracion['scenarios'].append(escenario_data)
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump(configuracion, f, ensure_ascii=False, indent=4)
                QtWidgets.QMessageBox.information(self, "Guardar Simulación", "La Simulación ha sido guardada exitosamente.")
            except Exception as e:
                QtWidgets.QMessageBox.critical(self, "Error", f"No se pudo guardar la Simulación: {e}")

    def cargar_configuracion(self):
        options = QtWidgets.QFileDialog.Options()
        filepath, _ = QtWidgets.QFileDialog.getOpenFileName(self, "Cargar Simulación", "",
                                                            "JSON Files (*.json);;All Files (*)", options=options)
        if filepath:
            # Agregar confirmación al usuario
            respuesta = QtWidgets.QMessageBox.question(
                self,
                "Cargar Simulación",
                "Al cargar una simulación, se eliminarán los escenarios y resultados actuales. ¿Desea continuar?",
                QtWidgets.QMessageBox.Yes | QtWidgets.QMessageBox.No
            )
            if respuesta != QtWidgets.QMessageBox.Yes:
                return
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    configuracion = json.load(f)

                # Antes de cargar la nueva configuración, limpiar escenarios y resultados
                # Limpiar escenarios
                self.scenarios.clear()
                self.scenarios_table.setRowCount(0)
                self.current_scenario = None
                self.selected_scenario_label.setText("Ninguno")

                # Limpiar resultados de simulación
                self.resultados_text_edit.clear()
                self.graficos_tab_widget.clear()

                # Limpiar eventos de riesgo
                self.eventos_riesgo.clear()
                self.eventos_table.setRowCount(0)

                # Cargar la nueva configuración
                self.num_simulaciones_var.setText(str(configuracion.get('num_simulaciones', 10000)))

                # Diccionario para mapear IDs antiguos a nuevos (para evitar conflictos)
                id_mapeo = {}

                # Cargar eventos de riesgo de la simulación principal
                for evento_data in configuracion.get('eventos_riesgo', []):
                    sev_opcion = evento_data['sev_opcion']
                    sev_input_method = evento_data.get('sev_input_method', 'min_mode_max') # Lee o usa default
                    sev_params_direct = evento_data.get('sev_params_direct', {}) # Lee o usa default
                    sev_minimo = evento_data['sev_minimo']
                    sev_mas_probable = evento_data['sev_mas_probable']
                    sev_maximo = evento_data['sev_maximo']

                    try:
                         dist_sev = generar_distribucion_severidad(
                            sev_opcion,
                            sev_minimo,
                            sev_mas_probable,
                            sev_maximo,
                            input_method=sev_input_method,
                            params_direct=sev_params_direct
                        )
                    except Exception as e:
                          print(f"Advertencia: No se pudo recrear dist_severidad para '{evento_data.get('nombre', 'N/A')}': {e}")
                          # Decide qué hacer: omitir evento, usar default, etc. Aquí lo omitimos por simplicidad.
                          continue # Saltar al siguiente evento si hay error

                    freq_opcion = evento_data['freq_opcion']
                    tasa = evento_data.get('tasa', None)
                    num_eventos = evento_data.get('num_eventos', None)
                    prob_exito = evento_data.get('prob_exito', None)
                    if tasa is not None:
                        tasa = float(tasa)
                    if num_eventos is not None:
                        num_eventos = int(num_eventos)
                    if prob_exito is not None:
                        prob_exito = float(prob_exito)
                    if 'eventos_padres' in evento_data and 'vinculos' not in evento_data:
                        vinculos = []
                        tipo = evento_data.get('tipo_dependencia', 'AND')
                        for padre_id in evento_data['eventos_padres']:
                            vinculos.append({'id_padre': padre_id, 'tipo': tipo})
                        evento_data['vinculos'] = vinculos

                    dist_freq = generar_distribucion_frecuencia(freq_opcion, tasa=tasa,
                                                                num_eventos_posibles=num_eventos, probabilidad_exito=prob_exito)
                    evento_data['dist_severidad'] = dist_sev
                    evento_data['dist_frecuencia'] = dist_freq

                    # Crear un nuevo ID para el evento y mapearlo
                    antiguo_id = evento_data['id']
                    nuevo_id = str(uuid.uuid4())
                    id_mapeo[antiguo_id] = nuevo_id
                    evento_data['id'] = nuevo_id

                    # Actualizar 'eventos_padres' con los nuevos IDs en caso de que existan en la simulación principal
                    eventos_padres_actualizados = []
                    for padre_id in evento_data.get('eventos_padres', []):
                        eventos_padres_actualizados.append(id_mapeo.get(padre_id, padre_id))
                    evento_data['eventos_padres'] = eventos_padres_actualizados

                    self.eventos_riesgo.append(evento_data)
                    row_position = self.eventos_table.rowCount()
                    self.eventos_table.insertRow(row_position)
                    self.eventos_table.setItem(row_position, 0, QtWidgets.QTableWidgetItem(evento_data['nombre']))

                # Actualizar IDs en los vínculos después de procesar todos los eventos
                for evento_data in self.eventos_riesgo:
                    if 'vinculos' in evento_data:
                        vinculos_actualizados = []
                        for vinculo in evento_data['vinculos']:
                            id_padre_antiguo = vinculo['id_padre']
                            id_padre_nuevo = id_mapeo.get(id_padre_antiguo, id_padre_antiguo)  # Usar ID antiguo si no hay mapeo
                            vinculos_actualizados.append({'id_padre': id_padre_nuevo, 'tipo': vinculo['tipo']})
                        evento_data['vinculos'] = vinculos_actualizados

                # Cargar escenarios
                for escenario_data in configuracion.get('scenarios', []):
                    scenario = Scenario(escenario_data['nombre'], escenario_data.get('descripcion', ''))
                    scenario.eventos_riesgo = []

                    # Nuevo diccionario de IDs para los eventos del escenario
                    id_mapeo_scenario = {}

                    for evento_data in escenario_data['eventos_riesgo']:
                        sev_opcion = evento_data['sev_opcion']
                        sev_minimo = evento_data['sev_minimo']
                        sev_mas_probable = evento_data['sev_mas_probable']
                        sev_maximo = evento_data['sev_maximo']
                        try:
                            dist_sev_esc = generar_distribucion_severidad(
                                evento_data['sev_opcion'],
                                evento_data.get('sev_minimo'),
                                evento_data.get('sev_mas_probable'),
                                evento_data.get('sev_maximo'),
                                input_method=evento_data.get('sev_input_method', 'min_mode_max'),
                                params_direct=evento_data.get('sev_params_direct', {})
                            )
                        except Exception as e:
                            print(f"Advertencia Escenario: No se pudo recrear dist_severidad para '{evento_data.get('nombre', 'N/A')}': {e}")
                            # Manejar error (ej. poner None y advertir al usuario)
                            dist_sev_esc = None # O manejar de otra forma

                        freq_opcion = evento_data['freq_opcion']
                        tasa = evento_data.get('tasa', None)
                        num_eventos = evento_data.get('num_eventos', None)
                        prob_exito = evento_data.get('prob_exito', None)
                        if tasa is not None:
                            tasa = float(tasa)
                        if num_eventos is not None:
                            num_eventos = int(num_eventos)
                        if prob_exito is not None:
                            prob_exito = float(prob_exito)
                        if 'eventos_padres' in evento_data and 'vinculos' not in evento_data:
                            vinculos = []
                            tipo = evento_data.get('tipo_dependencia', 'AND')
                            for padre_id in evento_data['eventos_padres']:
                                vinculos.append({'id_padre': padre_id, 'tipo': tipo})
                            evento_data['vinculos'] = vinculos

                        dist_freq = generar_distribucion_frecuencia(freq_opcion, tasa=tasa,
                                                                    num_eventos_posibles=num_eventos, probabilidad_exito=prob_exito)
                        evento_data['dist_severidad'] = dist_sev_esc
                        evento_data['dist_frecuencia'] = dist_freq

                        # Crear un nuevo ID para el evento y mapearlo
                        antiguo_id = evento_data['id']
                        nuevo_id = str(uuid.uuid4())
                        id_mapeo_scenario[antiguo_id] = nuevo_id
                        evento_data['id'] = nuevo_id

                        # Actualizar 'eventos_padres' con los nuevos IDs dentro del escenario
                        eventos_padres_actualizados = []
                        for padre_id in evento_data.get('eventos_padres', []):
                            if padre_id in id_mapeo_scenario:
                                eventos_padres_actualizados.append(id_mapeo_scenario[padre_id])
                            else:
                                # Si el evento padre está en la simulación principal
                                eventos_padres_actualizados.append(id_mapeo.get(padre_id, padre_id))
                        evento_data['eventos_padres'] = eventos_padres_actualizados

                        scenario.eventos_riesgo.append(evento_data)

                    self.scenarios.append(scenario)
                    row_position = self.scenarios_table.rowCount()
                    self.scenarios_table.insertRow(row_position)
                    self.scenarios_table.setItem(row_position, 0, QtWidgets.QTableWidgetItem(scenario.nombre))
                    self.scenarios_table.setItem(row_position, 1, QtWidgets.QTableWidgetItem(scenario.descripcion))

                    for evento_data in scenario.eventos_riesgo:
                        if 'vinculos' in evento_data:
                            vinculos_actualizados = []
                            for vinculo in evento_data['vinculos']:
                                id_padre_antiguo = vinculo['id_padre']
                                # Primero buscar en el mapeo del escenario, luego en el mapeo general
                                if id_padre_antiguo in id_mapeo_scenario:
                                    id_padre_nuevo = id_mapeo_scenario[id_padre_antiguo]
                                else:
                                    id_padre_nuevo = id_mapeo.get(id_padre_antiguo, id_padre_antiguo)
                                vinculos_actualizados.append({'id_padre': id_padre_nuevo, 'tipo': vinculo['tipo']})
                            evento_data['vinculos'] = vinculos_actualizados

                # Restaurar el escenario seleccionado actual, si existe
                current_scenario_name = configuracion.get('current_scenario_name')
                if current_scenario_name:
                    for scenario in self.scenarios:
                        if scenario.nombre == current_scenario_name:
                            self.current_scenario = scenario
                            self.selected_scenario_label.setText(scenario.nombre)
                            break

                QtWidgets.QMessageBox.information(self, "Cargar Simulación", "La Simulación ha sido cargada exitosamente.")
            except Exception as e:
                error_message = traceback.format_exc()
                QtWidgets.QMessageBox.critical(self, "Error", f"No se pudo cargar la Simulación: {e}\n{error_message}")

# Función principal
def main():
    app = QtWidgets.QApplication(sys.argv)
    window = RiskLabApp()
    window.show()
    sys.exit(app.exec_())

# Ejecutamos el programa principal
if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'numpy'